# Bütünler Üzerinde Gezinme — Granülarite Ablasyonu

Mevcut yöntem **yamalar ve cümle pencereleri** üzerinde geziniyor. Bu
notebook aynı deneyleri **bütün görseller** üzerinde tekrarlar; tek
değişen granülaritedir.

**Gezinme (danışman önerisi)**

```
p1  başlangıç yaması
p2  yama havuzundan p1'e en benzer yama        -> image(p2) zincire girer
p3  image(p2) içinden RASTGELE yama            -> aynı bölgeye saplanmayı önler
p4  yama havuzundan p3'e en benzer yama        -> image(p4) zincire girer
...
```

Zincir böylece **ana görsellerden** oluşur; iki görseli bağlayan bağ
"bir parçaları benziyor"dur. Aradaki rastgele kayma zincirin aynı
görsel motifinin etrafında dönmesini engeller.

**Bütün düzeyinde zorlu negatif.** Denklem 2'nin karşılığı:

| koşul | anlamı |
|---|---|
| parça benzerliği ≥ `PART_SIM_MIN` | zor olsun |
| `cos(bütün_A, bütün_B) < τ_s` | gerçekten negatif olsun |

İki görsel bütün olarak farklı ama karıştırılabilir bir bölge
paylaşıyorsa, biri diğerinin zorlu negatifi olur. Bütün düzeyinde en
yakın komşu araması böyle bir negatifi bulamaz.

**Koşullar** (ana deneylerle birebir aynı: 5 seed, aynı bölme, aynı
mimari, aynı erken durdurma)

| # | Kol |
|---|---|
| 0 | Dondurulmuş temel model, uyarlama yok |
| 1 | Vanilla InfoNCE, rastgele çiftler |
| 2 | Vanilla InfoNCE, zincir çiftleri |
| 3 | + HN-InfoNCE, mode conditioning |
| 4 | + SACL, hard window |
| 5 | + SACL, distance-weighted |

**Çıktılar** `ZoomInOut/whole_level/` altına yazılır; mevcut hiçbir
sonucun üzerine yazmaz.

**Süre:** bütün gömme ~10 dk, zincir üretimi ~5 dk, eğitim 5 kol × 5
seed ≈ 1.5-2 saat.

## 0. Ön kontrol

In [ ]:
# HÜCRE 0: gerekli dosyalar
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_BASE = "/content/drive/MyDrive/COSMOS"

GEREKLI = [
    (f"{DRIVE_BASE}/ds_sampled_image_datas_WH_224", True,  "2K gorseller"),
    (f"{DRIVE_BASE}/ZoomInOut/coco_image_dynamic_embeddings_new_WH_224_2048.pkl", False,
     "yama gommeleri (parca benzerligi icin)"),
    (f"{DRIVE_BASE}/ZoomInOut/v5_coco_captions_embedded.pkl", False, "caption gommeleri"),
    (f"{DRIVE_BASE}/ZoomInOut/simple_story_sentences_embeddings_new_tr_5000.pkl", False,
     "hikaye cumlesi gommeleri (W serisi: metin indeksi)"),
]
_eksik=[]
print(f"{'durum':<6}{'boyut':>10}  aciklama"); print("-"*58)
for _p,_d,_a in GEREKLI:
    _ok = os.path.isdir(_p) if _d else os.path.isfile(_p)
    _sz = "klasor" if (_ok and _d) else (f"{os.path.getsize(_p)/2**20:.0f} MB" if _ok else "-")
    if not _ok: _eksik.append(_a)
    print(f"{'  VAR' if _ok else 'EKSIK':<6}{_sz:>10}  {_a}")
assert not _eksik, f"Eksik: {_eksik}"

OUT = f"{DRIVE_BASE}/ZoomInOut/whole_level"; os.makedirs(OUT, exist_ok=True)
import torch
print(f"\nCikti  -> {OUT}")
print(f"GPU    : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'YOK'}")

## 1. Kurulum ve ortak ayarlar

In [ ]:
# HÜCRE 1: paketler, sabitler, bolme
!pip install -q datasets accelerate

import json, pickle, random, gc, copy, time, itertools
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk
from PIL import Image
from tqdm.auto import tqdm

SEED=42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# --- ana deneylerle AYNI hiperparametreler ---
EMBED_DIM=2048; HIDDEN=1024; TEMP=0.07
BATCH_SIZE=128; LR=1e-4; EPOCHS=75; PATIENCE=4; EVAL_EVERY=5
MODE_AUX_WEIGHT=0.1; LAMBDA_SEQ=0.1
LAMBDA_MULTIPOS=0.5; LAMBDA_RANK=0.5; RANK_MARGIN=0.1
CHAIN_BATCH_SIZE=16; SACL_MAX_LEN=65; POS_WINDOW=2; TAU_D=5.0
SEEDS=[42,123,777,2024,31415]

# --- butun duzeyine ozel ---
TAU_S        = 0.85   # cos(butun_A, butun_B) < TAU_S  -> gercekten negatif
PART_SIM_MIN = 0.55   # parca benzerligi esigi -> zor olsun
LOOPS        = 16     # zincir basina adim
NUM_CHAINS   = 600    # ana kosuyla ayni sekans butcesi
MIN_CHAIN    = 8

TRAIN_RATIO=0.75; VAL_RATIO=0.15

ds_img = load_from_disk(f"{DRIVE_BASE}/ds_sampled_image_datas_WH_224")
with open(f"{DRIVE_BASE}/ZoomInOut/coco_image_dynamic_embeddings_new_WH_224_2048.pkl","rb") as f:
    layout = pickle.load(f)
with open(f"{DRIVE_BASE}/ZoomInOut/v5_coco_captions_embedded.pkl","rb") as f:
    CAP_EMB = pickle.load(f)          # {coco_id: [(caption, emb), ...]}
print(f"gorsel {len(ds_img)} | yama {len(layout)} | caption {len(CAP_EMB)} gorsel")

def _coco_id(raw):
    if isinstance(raw,int): return raw
    s=str(raw).split(".")[0].lstrip("0") or "0"
    return int(s) if s.isdigit() else None

IMG_IDS = sorted({m["image_id"] for m in layout})
IID2COCO = {f"coco_image_{i}": _coco_id(ds_img[i]["image_id"]) for i in range(len(ds_img))}

# --- ana kosuyla AYNI bolme (make_image_split + carve_val_split) ---
_sh=IMG_IDS[:]; random.Random(42).shuffle(_sh)
_sp=int(len(_sh)*TRAIN_RATIO)
train_full, test_ids = set(_sh[:_sp]), set(_sh[_sp:])
_bt=sorted(train_full); random.Random(43).shuffle(_bt)
_sp2=int(len(_bt)*(1-VAL_RATIO))
train_ids, val_ids = set(_bt[:_sp2]), set(_bt[_sp2:])
print(f"bolme: train {len(train_ids)} | val {len(val_ids)} | test {len(test_ids)}")
assert (1275,225,500)==(len(train_ids),len(val_ids),len(test_ids)), "bolme ana kosuyla uyusmuyor!"

## 2. Bütün görsel gömmeleri

Yama çıkarımında global gömme hesaplanıyordu ama kaydedilmiyordu. Burada
her görsel için bir kez üretilip önbelleğe alınır. Qwen-2B hattıyla aynı
ön-işleme: `pad_to_square(224)` → gömme → L2.

In [ ]:
# HÜCRE 2: butun gorsel gommeleri (onbellekli, kesintiye dayanikli)
WHOLE_EMB_PATH = f"{OUT}/whole_image_embeddings.pkl"

def pad_to_square_and_resize(image, target=224):
    image=image.convert("RGB"); w,h=image.size; m=max(w,h)
    c=Image.new("RGB",(m,m),(128,128,128)); c.paste(image,((m-w)//2,(m-h)//2))
    return c.resize((target,target), Image.Resampling.LANCZOS)

if os.path.exists(WHOLE_EMB_PATH):
    with open(WHOLE_EMB_PATH,"rb") as f: WHOLE = pickle.load(f)
    print(f"Onbellek: {len(WHOLE)} gorsel")
else:
    WHOLE = {}

_kalan = [i for i in IMG_IDS if i not in WHOLE]
print(f"Uretilecek: {len(_kalan)}")
if _kalan:
    from transformers import AutoModel, AutoProcessor
    _mid="Qwen/Qwen3-VL-Embedding-2B"
    _m=AutoModel.from_pretrained(_mid, trust_remote_code=True, torch_dtype=torch.float16,
                                  device_map="auto", low_cpu_mem_usage=True)
    _p=AutoProcessor.from_pretrained(_mid, trust_remote_code=True)
    def _pool(h,am):
        if am[:,-1].sum()==am.shape[0]: return h[:,-1]
        return h[torch.arange(h.shape[0],device=h.device), am.sum(1)-1]
    IMG_INSTRUCTION="Represent this image patch for cross-modal retrieval with Turkish text."
    @torch.no_grad()
    def _embed_img(pil):
        msg=[{"role":"user","content":[{"type":"image","image":pil},
                                       {"type":"text","text":IMG_INSTRUCTION}]}]
        pr=_p.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
        inp=_p(text=[pr], images=[pil], return_tensors="pt", padding=True).to(device)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16) if device=="cuda" \
             else torch.autocast(device_type="cpu", enabled=False):
            o=_m(**inp, output_hidden_states=False, return_dict=True)
        e=_pool(o.last_hidden_state.float(), inp["attention_mask"])
        e=F.normalize(e,p=2,dim=-1).clamp(-1,1)
        return e[0].cpu().numpy().astype(np.float32)

    for k,iid in enumerate(tqdm(_kalan, desc="butun gorsel gomme")):
        idx=int(iid.split("_")[-1])
        WHOLE[iid]=_embed_img(pad_to_square_and_resize(ds_img[idx]["image"]))
        if (k+1)%200==0:
            with open(WHOLE_EMB_PATH,"wb") as f: pickle.dump(WHOLE,f)
    with open(WHOLE_EMB_PATH,"wb") as f: pickle.dump(WHOLE,f)
    del _m,_p; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

_v=np.stack([WHOLE[i] for i in IMG_IDS[:200]])
print(f"\n{len(WHOLE)} butun gomme hazir | boyut {_v.shape[1]} | "
      f"norm ort {np.linalg.norm(_v,axis=1).mean():.3f} (1.000 olmali)")
assert _v.shape[1]==EMBED_DIM

## 3. Bütün düzeyinde gezinme

Danışman algoritması. Parça benzerliği bağı kurar, rastgele kayma
saplanmayı önler. Zincir düğümleri **görsellerdir**.

In [ ]:
# HÜCRE 3: butun duzeyinde zincir uretimi
PATCH_EMB = np.stack([m["embedding"] for m in layout]).astype(np.float32)
PATCH_EMB /= (np.linalg.norm(PATCH_EMB,axis=1,keepdims=True)+1e-9)
PATCH_IID = np.array([m["image_id"] for m in layout])
PATCH_OBJ = np.array([m.get("objectness_score",1.0) for m in layout], dtype=np.float32)
_iid_list = IMG_IDS; _iid2code={v:k for k,v in enumerate(_iid_list)}
PATCH_CODE = np.fromiter((_iid2code[i] for i in PATCH_IID), dtype=np.int32, count=len(PATCH_IID))
by_img_idx = {}
for k,i in enumerate(PATCH_IID): by_img_idx.setdefault(i,[]).append(k)
print(f"yama matrisi {PATCH_EMB.shape}")

def _en_benzer_yama(q, yasak_kodlar, rng):
    """Havuzdaki EN BENZER yamayi dondur; yasakli gorsellerin yamalarini ele."""
    sim = PATCH_EMB @ q
    mask = np.zeros(len(_iid_list), dtype=bool)
    if yasak_kodlar: mask[np.fromiter(yasak_kodlar,dtype=np.int32)] = True
    sim = np.where(mask[PATCH_CODE], -999.0, sim)
    j = int(np.argmax(sim))
    return (None,None) if sim[j] <= -999.0 else (j, float(sim[j]))

def butun_zincir(start_iid, loops, rng):
    """p1 -> en benzer p2 -> image(p2)'den rastgele p3 -> en benzer p4 -> ...
    Dondurur: [(image_id, parca_benzerligi, hard_neg_mi), ...]"""
    havuz = by_img_idx[start_iid]
    obj = [k for k in havuz if PATCH_OBJ[k] >= 0.4] or havuz
    cur = rng.choice(obj)
    ziyaret = {_iid2code[start_iid]}
    zincir = [(start_iid, None, False)]
    for _ in range(loops):
        j, sim = _en_benzer_yama(PATCH_EMB[cur], ziyaret, rng)   # parca bagi
        if j is None: break
        yeni = PATCH_IID[j]
        # --- butun duzeyinde zorlu negatif kurali ---
        w_sim = float(WHOLE[zincir[-1][0]] @ WHOLE[yeni])
        hn = (sim >= PART_SIM_MIN) and (w_sim < TAU_S)
        zincir.append((yeni, sim, hn))
        ziyaret.add(_iid2code[yeni])
        havuz2 = by_img_idx[yeni]
        obj2 = [k for k in havuz2 if PATCH_OBJ[k] >= 0.4] or havuz2
        cur = rng.choice(obj2)                                   # rastgele kayma
    return zincir

CHAINS_PATH = f"{OUT}/whole_chains.pkl"
if os.path.exists(CHAINS_PATH):
    with open(CHAINS_PATH,"rb") as f: CHAINS = pickle.load(f)
    print(f"Onbellek: {len(CHAINS)} zincir")
else:
    rng = random.Random(1234)
    starts = sorted(train_ids); rng.shuffle(starts)
    CHAINS = []
    for i in tqdm(range(NUM_CHAINS), desc="butun zincir"):
        c = butun_zincir(starts[i % len(starts)], LOOPS, rng)
        if len(c) >= MIN_CHAIN: CHAINS.append(c)
    with open(CHAINS_PATH,"wb") as f: pickle.dump(CHAINS,f)

_dugum = sum(len(c) for c in CHAINS)
_hn = sum(1 for c in CHAINS for _,_,h in c if h)
_kapsam = len({iid for c in CHAINS for iid,_,_ in c} & train_ids)
print(f"\n{len(CHAINS)} zincir | {_dugum} dugum | zorlu negatif bagi {_hn} (%{_hn/_dugum*100:.1f})")
print(f"train havuzu kapsami: {_kapsam}/{len(train_ids)} (%{_kapsam/len(train_ids)*100:.1f})")

## 4. Çiftler: pozitifler, zorlu negatifler, rastgele kontrol

In [ ]:
# HÜCRE 4: egitim/val/test ciftleri
def _cap_emb(iid, rng):
    cid = IID2COCO.get(iid)
    if cid is None or cid not in CAP_EMB: return None
    return np.asarray(rng.choice(CAP_EMB[cid])[1], dtype=np.float32)

def zincir_ciftleri(chains, izin, rng):
    """Her dugum bir pozitif cift: (butun gorsel, kendi caption'i).
    Zincirde bir sonraki dugum zorlu negatif etiketliyse ona iliştirilir."""
    P=[]
    for c in chains:
        for k,(iid,sim,hn) in enumerate(c):
            if iid not in izin: continue
            ce=_cap_emb(iid,rng)
            if ce is None: continue
            hn_emb=None
            if k+1 < len(c) and c[k+1][2]:           # sonraki bag zorlu negatif
                hn_emb = WHOLE[c[k+1][0]]
            P.append({"img_embedding":WHOLE[iid], "txt_embedding":ce,
                      "hard_negative_embedding":hn_emb,
                      "hard_negative_mode":"part" if hn_emb is not None else None,
                      "chain_pos":k, "image_id":iid})
    return P

def rastgele_ciftler(izin, n, rng):
    """Kontrol: zincir hic kullanilmadan rastgele (butun gorsel, caption)."""
    P=[]; ids=sorted(izin); dene=0
    while len(P)<n and dene<n*30:
        dene+=1; iid=rng.choice(ids); ce=_cap_emb(iid,rng)
        if ce is None: continue
        P.append({"img_embedding":WHOLE[iid],"txt_embedding":ce,
                  "hard_negative_embedding":None,"hard_negative_mode":None,
                  "chain_pos":0,"image_id":iid})
    return P

def degerlendirme_ciftleri(izin, n, rng):
    """Her gorselden EN FAZLA 1 ornek -- ana kosudaki dedup mantigiyla ayni."""
    P=[]; ids=sorted(izin); rng.shuffle(ids)
    for iid in ids:
        ce=_cap_emb(iid,rng)
        if ce is None: continue
        P.append({"img_embedding":WHOLE[iid],"txt_embedding":ce})
        if len(P)>=n: break
    return P

_r=random.Random(7)
train_pairs  = zincir_ciftleri(CHAINS, train_ids, _r)
random_pairs = rastgele_ciftler(train_ids, len(train_pairs), random.Random(99))
val_pairs    = degerlendirme_ciftleri(val_ids, 150, random.Random(43))
test_pairs   = degerlendirme_ciftleri(test_ids, 400, random.Random(44))
_hn_n=sum(1 for p in train_pairs if p["hard_negative_embedding"] is not None)
print(f"train  {len(train_pairs)} cift (zorlu negatifli {_hn_n}, %{_hn_n/max(len(train_pairs),1)*100:.1f})")
print(f"random {len(random_pairs)} cift (kontrol)")
print(f"val    {len(val_pairs)} | test {len(test_pairs)}")

# --- SACL icin zincir kayitlari: dugumler BUTUN GORSEL, hepsi ayni kipten ---
chain_records=[]
for c in CHAINS:
    nodes=[{"step":k,"modality":0,"embedding":WHOLE[iid]}
           for k,(iid,_,_) in enumerate(c) if iid in train_ids]
    if len(nodes)>=MIN_CHAIN: chain_records.append(nodes)
print(f"SACL zincir kaydi: {len(chain_records)}")
print("\nNOT: butun duzeyinde TUM dugumler ayni kipten oldugu icin w=2 gercekten")
print("     iki adimi kapsar; yama duzeyindeki 'd=2 bos' sorunu burada YOKTUR.")

## 5. Mimari, kayıplar, eğitim altyapısı

Ana deneylerle **birebir aynı**: sıfır-başlatılmış residual adapter,
`SelfInferringHead`, HN-InfoNCE, SACL'in iki varyantı, aynı
hiperparametreler, aynı erken durdurma. Yalnızca veri değişiyor.

In [ ]:
# HÜCRE 5: mimari + kayiplar (ana notebook ile ayni)
class ResidualHead(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, context_dim=0, hidden=HIDDEN):
        super().__init__()
        self.mlp=nn.Sequential(nn.Linear(embed_dim+context_dim,hidden), nn.GELU(),
                                nn.LayerNorm(hidden), nn.Linear(hidden,embed_dim))
        nn.init.zeros_(self.mlp[-1].weight); nn.init.zeros_(self.mlp[-1].bias)
    def forward(self,x,context=None):
        inp = x if context is None else torch.cat([x,context],dim=-1)
        return F.normalize(x+self.mlp(inp), dim=-1)

class ProjectionHead(nn.Module):
    def __init__(self):
        super().__init__()
        self.img_proj=ResidualHead(); self.txt_proj=ResidualHead()
    def forward(self,img,txt): return self.img_proj(img), self.txt_proj(txt)

class SelfInferringHead(nn.Module):
    def __init__(self, num_modes=4):
        super().__init__()
        self.mode_table=nn.Embedding(num_modes, EMBED_DIM)
        self.mode_predictor=nn.Sequential(nn.Linear(EMBED_DIM*2,256), nn.GELU(),
                                           nn.Linear(256,num_modes))
        self.img_proj=ResidualHead(EMBED_DIM, context_dim=EMBED_DIM)
        self.txt_proj=ResidualHead(EMBED_DIM, context_dim=EMBED_DIM)
    def infer_mode(self,img,txt):
        logits=self.mode_predictor(torch.cat([img,txt],dim=-1))
        return F.softmax(logits,dim=-1) @ self.mode_table.weight, logits
    def forward(self,img,txt):
        mv,ml=self.infer_mode(img,txt)
        return self.img_proj(img,mv), self.txt_proj(txt,mv), ml
    def project_image_with_vec(self,img,mv): return self.img_proj(img,mv)

def plain_info_nce(img,txt,**kw):
    lab=torch.arange(img.size(0),device=img.device); lg=img@txt.T/TEMP
    return (F.cross_entropy(lg,lab)+F.cross_entropy(lg.T,lab))/2

def hn_info_nce(img,txt,z_hn=None,has_hn=None,**kw):
    lab=torch.arange(img.size(0),device=img.device)
    li=img@txt.T/TEMP; lt=txt@img.T/TEMP
    if z_hn is not None:
        s=(txt*z_hn).sum(-1,keepdim=True)/TEMP
        s=s.masked_fill(has_hn.unsqueeze(1)==0, float("-inf"))
        lt=torch.cat([lt,s],dim=1)
    return (F.cross_entropy(li,lab)+F.cross_entropy(lt,lab))/2

MODE_MAP={"semantic":0,"part":1}

class PairDataset(Dataset):
    def __init__(self,pairs): self.p=pairs
    def __len__(self): return len(self.p)
    def __getitem__(self,i):
        p=self.p[i]
        img=torch.tensor(np.asarray(p["img_embedding"],dtype=np.float32))
        txt=torch.tensor(np.asarray(p["txt_embedding"],dtype=np.float32))
        hn=p.get("hard_negative_embedding"); has=hn is not None
        hn_t=torch.tensor(np.asarray(hn,dtype=np.float32)) if has else torch.zeros_like(img)
        return {"img":img,"txt":txt,"hn":hn_t,
                "has_hn":torch.tensor(1.0 if has else 0.0),
                "mode_idx":torch.tensor(MODE_MAP.get(p.get("hard_negative_mode") or "semantic",0),
                                         dtype=torch.long)}

class ChainBatch:
    def __init__(self,emb,modality,mask,step_idx):
        self.emb=emb; self.modality=modality; self.mask=mask; self.step_idx=step_idx
    def to(self,d): return ChainBatch(self.emb.to(d),self.modality.to(d),
                                       self.mask.to(d),self.step_idx.to(d))

class ChainDataset(Dataset):
    def __init__(self,recs,max_len=SACL_MAX_LEN): self.c=recs; self.L=max_len
    def __len__(self): return len(self.c)
    def __getitem__(self,i):
        n=self.c[i][:self.L]
        emb=torch.zeros(self.L,EMBED_DIM); mod=torch.zeros(self.L,dtype=torch.long)
        msk=torch.zeros(self.L,dtype=torch.bool); st=torch.zeros(self.L,dtype=torch.long)
        for k,x in enumerate(n):
            emb[k]=torch.tensor(x["embedding"]); mod[k]=x["modality"]
            st[k]=x["step"]; msk[k]=True
        return emb,mod,msk,st

def chain_collate(b):
    return ChainBatch(torch.stack([x[0] for x in b]), torch.stack([x[1] for x in b]),
                       torch.stack([x[2] for x in b]), torch.stack([x[3] for x in b]))

def project_chain_self_infer(model,batch):
    B,L,D=batch.emb.shape; flat=batch.emb.view(B*L,D)
    mv,_=model.infer_mode(flat,flat)
    return F.normalize(model.img_proj(flat,mv),dim=-1).view(B,L,-1)

def _sacl_ortak(z,batch,agirlik_fn):
    B,L,Dp=z.shape; dev=z.device
    fz,fc,fs=[],[],[]
    for b in range(B):
        v=batch.mask[b]; n=int(v.sum().item())
        if n==0: continue
        fz.append(z[b,v]); fc.append(torch.full((n,),b,device=dev,dtype=torch.long))
        fs.append(batch.step_idx[b,v])
    fz=torch.cat(fz); fc=torch.cat(fc); fs=torch.cat(fs); N=fz.size(0)
    sim=fz@fz.T/TEMP
    ayni=fc.unsqueeze(0)==fc.unsqueeze(1)
    d=(fs.unsqueeze(0)-fs.unsqueeze(1)).abs()
    oz=torch.eye(N,dtype=torch.bool,device=dev)
    w=agirlik_fn(d,ayni,oz)
    lg=sim.masked_fill(oz,float("-inf"))
    lp=(lg-torch.logsumexp(lg,dim=1,keepdim=True)).masked_fill(oz,0.0)
    ws=w.sum(dim=1).clamp(min=1e-8)
    per=-(lp*w).sum(dim=1)/ws
    var=w.sum(dim=1)>1e-6
    l_mp=per[var].mean() if var.any() else torch.zeros((),device=dev)
    rt=[]
    for b in range(B):
        ib=(fc==b).nonzero(as_tuple=True)[0]
        if ib.numel()<3: continue
        for a in ib[torch.randperm(ib.numel(),device=dev)[:6]]:
            ot=ib[ib!=a]
            if ot.numel()<2: continue
            dd=(fs[ot]-fs[a]).abs(); o=torch.argsort(dd)
            if dd[o[-1]]<=dd[o[0]]: continue
            rt.append(F.relu(torch.tensor(RANK_MARGIN,device=dev)-
                             ((fz[a]*fz[ot[o[0]]]).sum()-(fz[a]*fz[ot[o[-1]]]).sum())/TEMP))
    l_rk=torch.stack(rt).mean() if rt else torch.zeros((),device=dev)
    return LAMBDA_MULTIPOS*l_mp + LAMBDA_RANK*l_rk

def sacl_hard(z,batch,w=POS_WINDOW):
    return _sacl_ortak(z,batch, lambda d,ayni,oz: (ayni&(d>0)&(d<=w)&(~oz)).float())
def sacl_cont(z,batch,tau=TAU_D):
    return _sacl_ortak(z,batch, lambda d,ayni,oz: torch.exp(-d.float()/tau)*(ayni&(~oz)).float())

@torch.no_grad()
def degerlendir(model, pairs, tip):
    model.eval()
    I=torch.tensor(np.stack([p["img_embedding"] for p in pairs])).to(device)
    T=torch.tensor(np.stack([p["txt_embedding"] for p in pairs])).to(device)
    if tip=="plain": zi,zt=model(I,T)
    else:
        mi,_=model.infer_mode(I,I); mt,_=model.infer_mode(T,T)
        zi=model.img_proj(I,mi); zt=model.txt_proj(T,mt)
    S=(zt@zi.T).cpu().numpy(); n=len(pairs)
    rank=(S>np.diag(S)[:,None]).sum(axis=1)+1
    return {"R@1":(rank==1).mean()*100,"R@5":(rank<=5).mean()*100,
            "R@10":(rank<=10).mean()*100,"MRR":(1.0/rank).mean()}
print("mimari + kayiplar hazir")

## Superseded: the first whole-image implementation

The arms run in the cells `HÜCRE 6`, `HÜCRE 7` and `HÜCRE 6b` below are **not** the ones reported in the paper and should not be used. That first implementation labelled every mined negative with a constant `"part"` mode, used a two-class `MODE_MAP` instead of the four-class one, and left the traversal unrestricted, so a training chain could reach a test image.

The reported whole-image results come from the `HÜCRE W2` to `HÜCRE W6` cells at the end of this notebook, which run the same algorithm as the patch-level notebook with the pool restricted to the training split. The cells above are kept because `HÜCRE W6` reuses the data loading of `HÜCRE 1`, the evaluation pairs of `HÜCRE 4` and the architecture of `HÜCRE 5`.


## 6. 6 kol × 5 seed, süre takipli

In [ ]:
# HÜCRE 6: tum kollari kosur (checkpoint'li, kesintiye dayanikli)
CKPT=f"{OUT}/whole_level_results.pkl"
SONUC = pickle.load(open(CKPT,"rb")) if os.path.exists(CKPT) else {}

KOLLAR = [
  ("1 Vanilla InfoNCE, rastgele ciftler", "plain", "plain", None,        random_pairs),
  ("2 Vanilla InfoNCE, zincir ciftleri",  "plain", "plain", None,        train_pairs),
  ("3 + HN-InfoNCE, mode conditioning",   "self",  "hn",    None,        train_pairs),
  ("4 + SACL, hard window",               "self",  "hn",    sacl_hard,   train_pairs),
  ("5 + SACL, distance-weighted",         "self",  "hn",    sacl_cont,   train_pairs),
]

def egit(tip, kayip, sacl_fn, pairs, seed):
    t0=time.time(); torch.manual_seed(seed)
    model=(ProjectionHead() if tip=="plain" else SelfInferringHead()).to(device)
    opt=torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    dl=DataLoader(PairDataset(pairs), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    cl=itertools.cycle(DataLoader(ChainDataset(chain_records), batch_size=CHAIN_BATCH_SIZE,
                                   shuffle=True, drop_last=True, collate_fn=chain_collate)) \
        if sacl_fn else None
    best=copy.deepcopy(model.state_dict()); best_r1=-1; best_ep=0; kotu=0; ep_run=0
    for ep in range(EPOCHS):
        ep_run=ep+1; model.train()
        for b in dl:
            img=b["img"].to(device); txt=b["txt"].to(device)
            hn=b["hn"].to(device); has=b["has_hn"].to(device); mi=b["mode_idx"].to(device)
            opt.zero_grad()
            if tip=="plain":
                zi,zt=model(img,txt); loss=plain_info_nce(zi,zt)
            else:
                mvi,ml=model.infer_mode(img,img); mvt,_=model.infer_mode(txt,txt)
                zi=model.img_proj(img,mvi); zt=model.txt_proj(txt,mvt)
                zhn=model.project_image_with_vec(hn,mvi) if has.sum()>0 else None
                loss=hn_info_nce(zi,zt,z_hn=zhn,has_hn=has)
                m=has.bool()
                if m.sum()>0: loss=loss+MODE_AUX_WEIGHT*F.cross_entropy(ml[m],mi[m])
                if sacl_fn is not None:
                    cb=next(cl).to(device)
                    loss=loss+LAMBDA_SEQ*sacl_fn(project_chain_self_infer(model,cb),cb)
            loss.backward(); opt.step()
        sch.step()
        if (ep+1)%EVAL_EVERY==0 or ep+1==EPOCHS:
            r=degerlendir(model,val_pairs,tip)["R@1"]
            if r>best_r1: best_r1, best_ep, kotu = r, ep+1, 0; best=copy.deepcopy(model.state_dict())
            else:
                kotu+=1
                if kotu>=PATIENCE: break
    model.load_state_dict(best)
    return model, {"epochs_run":ep_run,"best_epoch":best_ep,"best_val_r1":best_r1,
                    "elapsed_sec":time.time()-t0}

for ad,tip,kayip,sacl_fn,pairs in KOLLAR:
    SONUC.setdefault(ad,{})
    for sd in SEEDS:
        if sd in SONUC[ad]: continue
        print(f"\n=== {ad} | seed={sd} ===")
        m,meta=egit(tip,kayip,sacl_fn,pairs,sd)
        r=degerlendir(m,test_pairs,tip); meta.update(r)
        SONUC[ad][sd]=meta
        print(f"  epoch={meta['epochs_run']} en_iyi={meta['best_epoch']} "
              f"sure={meta['elapsed_sec']:.0f}sn  R@1={r['R@1']:.1f}%  MRR={r['MRR']:.3f}")
        pickle.dump(SONUC, open(CKPT,"wb")); del m; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
print(f"\nTamam -> {CKPT}")

## 7. Özet tablo

In [ ]:
# HÜCRE 7: ozet
# 0. kol: dondurulmus temel model (uyarlama yok) -- tek deterministik olcum
_I=np.stack([p["img_embedding"] for p in test_pairs]); _T=np.stack([p["txt_embedding"] for p in test_pairs])
_I/=np.linalg.norm(_I,axis=1,keepdims=True)+1e-9; _T/=np.linalg.norm(_T,axis=1,keepdims=True)+1e-9
_S=_T@_I.T; _rank=(_S>np.diag(_S)[:,None]).sum(axis=1)+1
FROZEN={"R@1":(_rank==1).mean()*100,"MRR":(1.0/_rank).mean()}

print(f"{'kol':<38}{'R@1 (%)':>16}{'MRR':>16}{'epoch':>8}{'sure(dk)':>10}")
print("-"*88)
print(f"{'0 Dondurulmus temel model':<38}{FROZEN['R@1']:>16.1f}{FROZEN['MRR']:>16.3f}"
      f"{'--':>8}{'--':>10}")
ozet={"frozen":FROZEN,"kollar":{}}
for ad,_,_,_,_ in KOLLAR:
    d=SONUC.get(ad,{})
    if len(d)<len(SEEDS): print(f"{ad:<38}{'(eksik: '+str(len(d))+'/5 seed)':>16}"); continue
    r1=np.array([d[s]["R@1"] for s in SEEDS]); mr=np.array([d[s]["MRR"] for s in SEEDS])
    ep=np.mean([d[s]["epochs_run"] for s in SEEDS]); sec=np.mean([d[s]["elapsed_sec"] for s in SEEDS])
    print(f"{ad:<38}{f'{r1.mean():.2f} ± {r1.std(ddof=1):.2f}':>16}"
          f"{f'{mr.mean():.3f} ± {mr.std(ddof=1):.3f}':>16}{ep:>8.1f}{sec/60:>10.1f}")
    ozet["kollar"][ad]={"R@1_mean":r1.mean(),"R@1_std":r1.std(ddof=1),
                         "MRR_mean":mr.mean(),"MRR_std":mr.std(ddof=1),
                         "epoch_mean":ep,"sure_dk":sec/60,"seed_r1":r1.tolist()}
print("-"*88)
print(f"sans duzeyi: {100/len(test_pairs):.2f}%   |   test n={len(test_pairs)}")

# --- ana deneylerle ayni t-testleri ---
from scipy import stats
def _t(a,b,ad):
    if a not in ozet["kollar"] or b not in ozet["kollar"]: return
    x=np.array(ozet["kollar"][a]["seed_r1"]); y=np.array(ozet["kollar"][b]["seed_r1"])
    t,p=stats.ttest_rel(x,y)
    print(f"  {ad:<52} t={t:+.3f}  p={p:.4f}")
print("\npaired t-test (5 seed):")
_t("2 Vanilla InfoNCE, zincir ciftleri","1 Vanilla InfoNCE, rastgele ciftler",
   "zincir ciftleri vs rastgele ciftler")
_t("3 + HN-InfoNCE, mode conditioning","2 Vanilla InfoNCE, zincir ciftleri",
   "HN-InfoNCE vs vanilla (ayni ciftler)")
_t("4 + SACL, hard window","2 Vanilla InfoNCE, zincir ciftleri",
   "SACL hard window vs cift-duzeyi taban")
_t("5 + SACL, distance-weighted","4 + SACL, hard window",
   "distance-weighted vs hard window")

pickle.dump(ozet, open(f"{OUT}/whole_level_summary.pkl","wb"))
print(f"\nKaydedildi -> {OUT}/whole_level_summary.pkl")

## Sonuçları nasıl yorumlamalı

Bu tablo, ana deneylerdeki Tablo II'nin **bütün düzeyi** karşılığıdır.
Aynı bölme, aynı 5 seed, aynı mimari, aynı erken durdurma. Tek fark
granülarite.

**Karşılaştırırken dikkat:** iki tablo farklı test kümelerinde ölçülür.
Yama düzeyinde test çiftleri motorun ürettiği (yama, caption) çiftleri,
burada ise (bütün görsel, caption) çiftleri. Dolayısıyla mutlak R@1
değerleri doğrudan kıyaslanmamalı; **dondurulmuş temel modele göre
kazanım** üzerinden karşılaştırılmalıdır. Her iki tabloda da 0. satır
bu referansı verir.

**Makaleye yazarken** bu bir granülarite ablasyonudur: parça düzeyi
gezinme ile bütün düzeyi gezinmenin aynı veri ve aynı protokol altında
karşılaştırılması. Hangi yöne çıkarsa çıksın raporlanabilir bir
sonuçtur.

---
### Bu oturumda ne yapılacak

Hücreleri **sırayla** çalıştır. 6b dışındaki her şey önbellekten gelir:

| Hücre | Beklenen |
|---|---|
| 0 | Drive bağlanır, üç dosya VAR görünmeli |
| 1 | pip install + Drive'dan üç dosya okunur |
| 2 | **`Onbellek: N gorsel`** yazmalı. `Uretilecek: 2000` yazarsa DURDUR — gömme önbelleği kaybolmuş, Qwen yüklenir |
| 3 | zincirler önbellekten |
| 4 | çiftler deterministik olarak yeniden kurulur, aynı çıkar |
| 5 | mimari tanımları |
| 6 | altı kol da checkpoint'te bulunup atlanır |
| **6b** | eksik iki kol koşar, ~3 dk |

Toplam 8-12 dakika, büyük kısmı Drive'dan okuma.

## 6b. Eksik İki Kol: 2×2 Ayrıştırması

Yama düzeyinde "yalnız HN itme" ve "yalnız mod koşullama" kolları koşuldu ve
kazancın tümüyle mod etiketlerinden geldiği bulundu. Aynı ayrıştırma bütün
düzeyinde de yapılmadığı için iki tablo şu an simetrik değil. Bu hücre eksik
iki kolu ekler, mevcut altı kola dokunmaz, ve altı kolun MRR değerlerini de
birlikte basar.

In [ ]:
# HÜCRE 6b: bütün düzeyinde EKSİK İKİ KOL -- 2x2 ayrıştırmasını tamamlar
# ---------------------------------------------------------------------------
# Yama duzeyinde bugun "yalniz HN itme" ve "yalniz mod kosullama" kollari
# kosuldu ve kazancin tumuyle mod etiketlerinden geldigi bulundu. Ayni
# ayristirma butun duzeyinde de yapilmali, yoksa iki tablo simetrik olmaz.
# Mevcut alti kola DOKUNMAZ, ayni CKPT'ye ekler. ~3 dk.
import os, time, copy, gc, itertools, pickle
import numpy as np, torch, torch.nn.functional as F
from torch.utils.data import DataLoader

CKPT = f"{OUT}/whole_level_results.pkl"
SONUC = pickle.load(open(CKPT, "rb")) if os.path.exists(CKPT) else {}

def egit2(varyant, pairs, seed):
    """varyant: 'hn_only'   -> ProjectionHead + hn_info_nce, mod kosullama YOK
                'mode_only' -> SelfInferringHead + plain_info_nce + mod yardimci
                               kaybi, HN itme YOK"""
    t0 = time.time(); torch.manual_seed(seed)
    tip = "plain" if varyant == "hn_only" else "self"
    model = (ProjectionHead() if tip == "plain" else SelfInferringHead()).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    dl = DataLoader(PairDataset(pairs), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    best = copy.deepcopy(model.state_dict()); best_r1 = -1; best_ep = 0; kotu = 0; ep_run = 0
    for ep in range(EPOCHS):
        ep_run = ep + 1; model.train()
        for b in dl:
            img = b["img"].to(device); txt = b["txt"].to(device)
            hn = b["hn"].to(device); has = b["has_hn"].to(device); mi = b["mode_idx"].to(device)
            opt.zero_grad()
            if varyant == "hn_only":
                zi, zt = model(img, txt)
                zhn = model.img_proj(hn) if has.sum() > 0 else None
                loss = hn_info_nce(zi, zt, z_hn=zhn, has_hn=has)
            else:
                mvi, ml = model.infer_mode(img, img); mvt, _ = model.infer_mode(txt, txt)
                zi = model.img_proj(img, mvi); zt = model.txt_proj(txt, mvt)
                loss = plain_info_nce(zi, zt)                      # HN itme YOK
                m = has.bool()
                if m.sum() > 0:
                    loss = loss + MODE_AUX_WEIGHT * F.cross_entropy(ml[m], mi[m])
            loss.backward(); opt.step()
        sch.step()
        if (ep + 1) % EVAL_EVERY == 0 or ep + 1 == EPOCHS:
            r = degerlendir(model, val_pairs, tip)["R@1"]
            if r > best_r1: best_r1, best_ep, kotu = r, ep + 1, 0; best = copy.deepcopy(model.state_dict())
            else:
                kotu += 1
                if kotu >= PATIENCE: break
    model.load_state_dict(best)
    return model, tip, {"epochs_run": ep_run, "best_epoch": best_ep,
                        "best_val_r1": best_r1, "elapsed_sec": time.time() - t0}

EK_KOLLAR = [("2b ZoomInOut, yalniz HN itme",        "hn_only"),
             ("2c ZoomInOut, yalniz mod kosullama",  "mode_only")]

for ad, varyant in EK_KOLLAR:
    SONUC.setdefault(ad, {})
    for sd in SEEDS:
        if sd in SONUC[ad]: continue
        print(f"\n=== {ad} | seed={sd} ===")
        m, tip, meta = egit2(varyant, train_pairs, sd)
        r = degerlendir(m, test_pairs, tip); meta.update(r)
        SONUC[ad][sd] = meta
        print(f"  epoch={meta['epochs_run']} en_iyi={meta['best_epoch']} "
              f"sure={meta['elapsed_sec']:.0f}sn  R@1={r['R@1']:.1f}%  MRR={r['MRR']:.3f}")
        pickle.dump(SONUC, open(CKPT, "wb")); del m; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

# ---------- birlesik ozet: butun kollar, sirali ----------
print(f"\n{'kol':<40}{'R@1 (%)':>17}{'MRR':>17}{'dk':>7}{'epoch':>7}")
print("-" * 88)
for ad in sorted(SONUC):
    d = SONUC[ad]
    if len(d) < len(SEEDS): print(f"{ad:<40}{f'({len(d)}/{len(SEEDS)})':>17}"); continue
    r1 = np.array([d[s]["R@1"] for s in SEEDS]); mr = np.array([d[s]["MRR"] for s in SEEDS])
    print(f"{ad:<40}{f'{r1.mean():.2f} ± {r1.std(ddof=1):.2f}':>17}"
          f"{f'{mr.mean():.3f} ± {mr.std(ddof=1):.3f}':>17}"
          f"{np.mean([d[s]['elapsed_sec'] for s in SEEDS])/60:>7.1f}"
          f"{np.mean([d[s]['epochs_run'] for s in SEEDS]):>7.1f}")
print("-" * 88)
print(f"seed: {SEEDS}  |  std: ddof=1")

# ---------- 2x2 + t-testler ----------
from scipy import stats
def al(ad): return np.array([SONUC[ad][s]["R@1"] for s in SEEDS]) if len(SONUC.get(ad, {})) == len(SEEDS) else None
A  = al("2 Vanilla InfoNCE, zincir ciftleri")
B  = al("2b ZoomInOut, yalniz HN itme")
Cc = al("2c ZoomInOut, yalniz mod kosullama")
D  = al("3 + HN-InfoNCE, mode conditioning")
if all(x is not None for x in (A, B, Cc, D)):
    print(f"\n{'':<24}{'HN itme YOK':>16}{'HN itme VAR':>16}")
    print(f"{'mod kosullama YOK':<24}{A.mean():>16.2f}{B.mean():>16.2f}")
    print(f"{'mod kosullama VAR':<24}{Cc.mean():>16.2f}{D.mean():>16.2f}")
    for x, y, et in [(B, A, "HN itme, mod YOKken"), (D, Cc, "HN itme, mod VARken"),
                     (Cc, A, "mod kosullama, HN YOKken"), (D, B, "mod kosullama, HN VARken")]:
        t, p = stats.ttest_rel(x, y)
        print(f"  {et:<28}{x.mean()-y.mean():+7.2f} puan   t={t:+7.3f}  p={p:.4f}")


---
## Birebir aynı yöntem, yama yerine tam fotoğraflar (W serisi)

Bu seri, yama düzeyinde önerilen ve çalıştırılan yöntemin **aynısını** görsel
düğüm olarak tam fotoğraflar üzerinde çalıştırır. Böylece iki düzey birebir
karşılaştırılabilir bir tabloda verilebilir.

Aynı kalan her şey: `MultimodalSpiralEngine`, dört adım tipi
(`img_to_txt`, `txt_shift`, `txt_to_img`, `img_shift`), aynı hikâye cümlesi
pencereleri ve aynı metin indeksi, `top_k=8` ve `alpha=0.03` hub düzeltmesi,
`min_win_size=2`, `SIM_THRESHOLD=0.55`, `L=16` döngü, 600 sekans,
üç yüzey modunun yarışması, percentile-rank kalibrasyonu,
`h_m = yüzdelik(ŝ_m) − s_sem`, eşiklerin kalibrasyonla belirlenmesi,
`tau_s=0.85`, pozitif metnin
görselin kendi COCO caption'ı ile değiştirilmesi, mod koşullamalı residual
adapter, `hn_info_nce`, dört sınıflı mod haritası, `lambda_aux=0.1` ve aynı
eğitim protokolü.

Yöntemin tanımı gereği değişmesi gereken iki yer vardır, ikisi de aynı
nedenden kaynaklanır: bir tam fotoğrafın "başka bir parçası" yoktur.

1. `img_shift` adımının aday havuzu. Yama düzeyinde bu adım aynı sahnenin
   başka bir yamasına geçer; bütün düzeyinde adaylar havuzdaki diğer tam
   fotoğraflardır. Yarışma kuralı, eşikler ve etiketleme aynen korunur.
2. `build_pairs_from_sequences` içindeki `shift_step.image_id ==
   img_step.image_id` koşulu. Yama düzeyinde `img_shift` aynı görselde kaldığı
   için eşitlik doğrudur; bütün düzeyinde farklı bir görsele geçildiği için
   koşul `!=` olur. Koşul olduğu gibi bırakılsa hiçbir hard negative
   iliştirilemezdi.

Bunların dışında iki küçük uygulama farkı vardır: yama düzeyindeki
`secrets.choice` yerine tohumlanmış `rng` kullanılır (zincirler yeniden
üretilebilir olsun diye) ve `img_shift` hedefi `visited_image_ids` kümesine de
eklenir (yama düzeyinde etkisiz bir satır, bütün düzeyinde gerekli).

### Eşik, bu düzeyde yeniden kalibre edilir

Yama düzeyinde `tau_h = 0.1` yöntemin sabiti değildir; notebook'un HÜCRE E1'i
eşikleri tarayıp %15-30 verim veren ve mod dağılımı dengeli olan değeri
seçmektedir, HÜCRE 18 de öznitelik ölçeği değiştiğinde eski eşiğin anlamsız
kalacağını belirtmektedir. Bütün görsellerde ölçek gerçekten değişmektedir:
şekil betimleyicisi rastgele görsel çiftlerinde ortalama 0.9898 benzerlik ve
0.0153 standart sapma vermekte, yani neredeyse sabit kalmaktadır. Bu nedenle
yüzdelik-rank hemen her adayı dağılımın tepesine taşımakta ve `h` değeri
`1 − s_sem` civarına oturmaktadır. Eşik W4c ile yeniden kalibre edilir;
kalibrasyon adımı yöntemin kendisinin parçasıdır.

`img_shift` aday havuzunun boyutu da ölçümle belirlenir: W4b bu boyutu yama
düzeyindeki havuzun ölçülen ortalamasına, yani aynı görselin `objectness`
değeri 0.4 üzerindeki yama sayısının ortalamasına eşitler.

Değerlendirme iki kümede birlikte raporlanır: motorla üretilen test çiftleri
(yama düzeyiyle aynı yöntem) ve mevcut bütün düzeyi kollarıyla
karşılaştırılabilir olan basit test kümesi.

Gerekli ek dosya: `ZoomInOut/simple_story_sentences_embeddings_new_tr_5000.pkl`

### Çalıştırma sırası

HÜCRE 0 → 1 → 2 → 5, sonra W2 → W3 → W4a → W4b → W4b2 → W4c → W5 → W6.

HÜCRE 5 mimariyi, kayıpları ve `degerlendir` fonksiyonunu tanımladığı için
W6 tarafından kullanılır, atlanamaz. HÜCRE 3, 4, 6, 6b ve 7 ise W serisi için
gerekmez; W6 `test_pairs` ile `_cap_emb`'i kendisi kurar. Mevcut bütün düzeyi
sonuçlarını yeniden üretmek istenirse HÜCRE 3, 4, 6 ve 6b de çalıştırılır.

Önbellekler korunur: `whole_image_embeddings.pkl` (HÜCRE 2),
`whole_cv_feats.pkl` ve `whole_mode_calib.pkl` (W3) diskte varsa yeniden
hesaplanmaz. Eşik veya aday havuzu değiştirilirse yalnızca
`butun_birebir_pairs.pkl` silinir.
W4b2, `query_text`, `query_image` ve `find_hard_negative_text` çağrılarını
vektörleştirir. Bu üç fonksiyon her çağrıda 5000 metin yaması üzerinde Python
döngüsü kurup her yamada `text.split()` çalıştırıyordu ve sekans başına
yaklaşık 21 saniye harcıyordu. Kalite ve `win_size` filtreleri sabit olduğu
için bir kez önceden hesaplanır. Hücre, yavaş ve hızlı yolun aynı sonucu
verdiğini 75 sorguyla doğrular ve farklıysa durur.

In [ ]:
# HÜCRE W2: dataclass'lar + HybridMultimodalIndex
# Yama duzeyindeki notebook'un HUCRE 2 ve HUCRE 3'unden BIREBIR kopya.
# Tek satir degisiklik yok -- fark sadece indekse NE konuldugunda.
from dataclasses import dataclass, field
from typing import List, Tuple, Optional, Set
import numpy as np, re, secrets

@dataclass
class ImagePatchMeta:
    image_id:         str
    patch_id:         str
    bbox:             list
    embedding:        np.ndarray
    objectness_score: float = 1.0
    type:             str   = "object"
    window_type:      str   = ""
    area:             float = 0.0
    attention_score:  float = 0.0
    color_feat:       Optional[np.ndarray] = None
    shape_feat:       Optional[np.ndarray] = None
    texture_feat:     Optional[np.ndarray] = None

    def iou(self, other: "ImagePatchMeta") -> float:
        x1 = max(self.bbox[0], other.bbox[0])
        y1 = max(self.bbox[1], other.bbox[1])
        x2 = min(self.bbox[2], other.bbox[2])
        y2 = min(self.bbox[3], other.bbox[3])
        inter = max(0, x2 - x1) * max(0, y2 - y1)
        if inter == 0:
            return 0.0
        a1 = (self.bbox[2] - self.bbox[0]) * (self.bbox[3] - self.bbox[1])
        a2 = (other.bbox[2] - other.bbox[0]) * (other.bbox[3] - other.bbox[1])
        return inter / (a1 + a2 - inter + 1e-10)


@dataclass
class TextPatchMeta:
    doc_id:         str
    patch_id:       str
    text_content:   str
    embedding:      np.ndarray
    win_size:       int   = 2
    win_label:      str   = "w2"
    richness_score: float = 0.0
    sent_range:     tuple = field(default_factory=lambda: (0, 0))
    lexical_feat:   Optional[np.ndarray] = None  # TF-IDF -- metin-tarafı hard-negative madenciliği için


@dataclass
class HybridStepRecord:
    step:            int
    action:          str
    modality:        str
    image_id:        Optional[str]   = None
    patch_id:        Optional[str]   = None
    bbox:            Optional[list]  = None
    doc_id:          Optional[str]   = None
    text_content:    Optional[str]   = None
    similarity:      float           = 0.0
    similarity_mode: str             = "semantic"
    verified_label:  str             = "positive"
    # Metin-tarafı hard-negative (sadece img_to_txt/txt_to_img adımlarında dolu olur):
    hard_negative_txt_patch_id: Optional[str]   = None
    hard_negative_txt_sim:      Optional[float] = None
    hard_negative_txt_mode:     Optional[str]   = None


class HybridMultimodalIndex:
    def __init__(self):
        self.image_patches: List[ImagePatchMeta] = []
        self.text_patches:  List[TextPatchMeta]  = []
        self._img_matrix:   Optional[np.ndarray] = None
        self._txt_matrix:   Optional[np.ndarray] = None
        self._lex_matrix:   Optional[np.ndarray] = None
        self._img_dirty = True
        self._txt_dirty = True
        self._lex_dirty = True

    def _is_quality_text(self, text: str) -> bool:
        if not text or len(text.split()) < 5:
            return False
        if text.strip().endswith("?"):
            return False
        BAGLAC = {"ve", "ama", "fakat", "lakin", "çünkü", "ancak",
                  "ile", "ya", "da", "de", "ki", "bu", "şu", "o",
                  "diye", "hem", "ne", "ah", "oh", "hmm"}
        ilk = text.split()[0].lower().rstrip(".,!?")
        if ilk in BAGLAC:
            return False
        if text.startswith('"') or text.startswith('-') or '—' in text:
            return False
        return True

    def add_image_patch(self, patch: ImagePatchMeta):
        for attr in ("embedding", "color_feat", "shape_feat", "texture_feat"):
            feat = getattr(patch, attr, None)
            if feat is not None:
                n = np.linalg.norm(feat)
                if n > 0:
                    setattr(patch, attr, feat / n)
        self.image_patches.append(patch)
        self._img_dirty = True

    def add_text_patch(self, patch: TextPatchMeta):
        n = np.linalg.norm(patch.embedding)
        if n > 0:
            patch.embedding = patch.embedding / n
        if patch.lexical_feat is not None:
            ln = np.linalg.norm(patch.lexical_feat)
            if ln > 0:
                patch.lexical_feat = patch.lexical_feat / ln
        self.text_patches.append(patch)
        self._txt_dirty = True
        self._lex_dirty = True

    def _build_matrices(self):
        if self._img_dirty and self.image_patches:
            self._img_matrix = np.stack([p.embedding for p in self.image_patches], axis=0).astype(np.float32)
            self._img_dirty = False
        if self._txt_dirty and self.text_patches:
            self._txt_matrix = np.stack([p.embedding for p in self.text_patches], axis=0).astype(np.float32)
            self._txt_dirty = False
        if self._lex_dirty and self.text_patches and all(p.lexical_feat is not None for p in self.text_patches):
            self._lex_matrix = np.stack([p.lexical_feat for p in self.text_patches], axis=0).astype(np.float32)
            self._lex_dirty = False

    def query_text(self, query_emb, exclude_doc_ids=None, threshold=-1.0, min_win_size=1, top_k=1):
        """v5i: top_k>1 verilirse, tek-en-iyi yerine EN IYI top_k adayi
        (agirlikli skora gore sirali) dondurur -- motor bunlar arasindan
        cesitlilik-farkinda secim yapabilsin diye (bkz. MultimodalSpiralEngine)."""
        self._build_matrices()
        if self._txt_matrix is None or not self.text_patches:
            return []
        exclude_doc_ids = exclude_doc_ids or set()
        q = query_emb / (np.linalg.norm(query_emb) + 1e-9)
        raw = np.clip(self._txt_matrix @ q, -1.0, 1.0)
        WIN_WEIGHT = {1: 0.70, 2: 1.05, 3: 1.15}
        weighted = raw.copy()
        for i, p in enumerate(self.text_patches):
            if p.doc_id in exclude_doc_ids or p.win_size < min_win_size or not self._is_quality_text(p.text_content):
                weighted[i] = -999.0
            else:
                weighted[i] = weighted[i] * WIN_WEIGHT.get(p.win_size, 1.0)
        if top_k <= 1:
            best_idx = int(np.argmax(weighted))
            if weighted[best_idx] <= -999.0:
                return []
            return [(self.text_patches[best_idx], float(raw[best_idx]))]
        order = np.argsort(-weighted)[:top_k]
        return [(self.text_patches[i], float(raw[i])) for i in order if weighted[i] > -999.0]

    def query_image(self, query_emb, exclude_image_ids=None, top_k=1):
        """v5i: top_k>1 verilirse, tek-en-iyi yerine EN IYI top_k adayi dondurur."""
        self._build_matrices()
        if self._img_matrix is None or not self.image_patches:
            return []
        exclude_image_ids = exclude_image_ids or set()
        q = query_emb / (np.linalg.norm(query_emb) + 1e-9)
        raw = np.clip(self._img_matrix @ q, -1.0, 1.0)
        TYPE_WEIGHT = {"object": 1.10, "background": 0.92}
        weighted = raw.copy()
        for i, p in enumerate(self.image_patches):
            if p.image_id in exclude_image_ids:
                weighted[i] = -999.0
            else:
                weighted[i] = weighted[i] * TYPE_WEIGHT.get(p.type, 1.0)
        if top_k <= 1:
            best_idx = int(np.argmax(weighted))
            if weighted[best_idx] <= -999.0:
                return []
            return [(self.image_patches[best_idx], float(raw[best_idx]))]
        order = np.argsort(-weighted)[:top_k]
        return [(self.image_patches[i], float(raw[i])) for i in order if weighted[i] > -999.0]

    def text_patches_of(self, doc_id):
        return [p for p in self.text_patches if p.doc_id == doc_id]

    def image_patches_of(self, image_id):
        return [p for p in self.image_patches if p.image_id == image_id]

    def image_ids(self):
        # v5h: SIRALI donduruluyor -- set() sirasi Python surecleri arasinda
        # hash-randomizasyonu yuzunden DEGISEBILIR, bu da train/test
        # bolunmesinin oturumdan oturuma FARKLI cikmasina (ve sizintiya)
        # yol aciyordu.
        return sorted({p.image_id for p in self.image_patches})

    def find_hard_negative_text(self, anchor_patch: "TextPatchMeta", exclude_doc_ids,
                                 hardness_min: float = 0.05, hard_neg_sem_thresh: float = 0.85):
        """Fikir 2: metin-tarafı hard-negative arama. anchor_patch ile YÜZEYDE
        (TF-IDF/leksik) benzer ama SEMANTİK olarak farklı, BAŞKA bir dokümandan
        bir metin adayı arar -- görsel tarafındaki renk/şekil/doku mantığının
        metin karşılığı. Traversal'ı ETKİLEMEZ, sadece paralel bir alan olarak
        (img_to_txt adımına) eklenir; txt_shift'in (chain tutarlılığı için
        kritik) semantik-en-yakın seçim mantığına dokunmaz.
        """
        self._build_matrices()
        if self._lex_matrix is None or anchor_patch.lexical_feat is None:
            return None
        surface_all = np.clip(self._lex_matrix @ anchor_patch.lexical_feat, -1.0, 1.0)
        sem_all = np.clip(self._txt_matrix @ anchor_patch.embedding, -1.0, 1.0)
        hardness_all = surface_all - sem_all

        best_idx, best_hardness = -1, -999.0
        for i, p in enumerate(self.text_patches):
            if p.doc_id in exclude_doc_ids or p.doc_id == anchor_patch.doc_id:
                continue
            if not self._is_quality_text(p.text_content):
                continue
            if hardness_all[i] > best_hardness:
                best_idx, best_hardness = i, hardness_all[i]

        if best_idx == -1 or best_hardness < hardness_min or sem_all[best_idx] >= hard_neg_sem_thresh:
            return None
        cand = self.text_patches[best_idx]
        return cand.patch_id, float(surface_all[best_idx]), "lexical"

In [ ]:
# HÜCRE W3: butun gorseller icin renk/sekil/doku betimleyicileri + mod kalibrasyonu
# ---------------------------------------------------------------------------
# Formuller yama duzeyindeki HUCRE 5b ile BIREBIR AYNI. Tek fark: kirpma yok,
# betimleyiciler gorselin tamami uzerinde hesaplaniyor; aspect_ratio da
# gorselin kendi W/H orani.
#   renk   : HSV histogram  (H 16 + S 8 + V 8  = 32 boyut)
#   doku   : Sobel buyukluk histogrami 32 + LBP uniform 10 = 42 boyut
#   sekil  : Canny kenar histogrami 32 + 7 log-Hu + aspect ratio = 40 boyut
# Hepsi L2 normalize.
import os, pickle, random
import numpy as np
from tqdm.auto import tqdm

try:
    import cv2
except ImportError:
    get_ipython().system("pip install -q opencv-python-headless")
    import cv2
from skimage.feature import local_binary_pattern

WCV_PATH   = f"{OUT}/whole_cv_feats.pkl"
WCAL_PATH  = f"{OUT}/whole_mode_calib.pkl"
MODLAR     = ("color", "shape", "texture")

def _butun_betimleyici(pil_img):
    """Tek bir butun gorsel icin (renk, sekil, doku) betimleyicileri."""
    img_bgr = cv2.cvtColor(np.array(pil_img.convert("RGB")), cv2.COLOR_RGB2BGR)
    H_img, W_img = img_bgr.shape[:2]

    # --- RENK: HSV histogrami ---
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    hist_h = cv2.calcHist([hsv], [0], None, [16], [0, 180])
    hist_s = cv2.calcHist([hsv], [1], None, [8],  [0, 256])
    hist_v = cv2.calcHist([hsv], [2], None, [8],  [0, 256])
    color_f = np.concatenate([hist_h, hist_s, hist_v]).flatten().astype(np.float32)
    color_f /= (np.linalg.norm(color_f) + 1e-10)

    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

    # --- DOKU: Sobel histogrami + LBP histogrami ---
    sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    mag = np.sqrt(sobelx ** 2 + sobely ** 2)
    hist_txt, _ = np.histogram(mag, bins=32, range=(0, 255))
    lbp = local_binary_pattern(gray, P=8, R=1, method="uniform")
    hist_lbp, _ = np.histogram(lbp, bins=10, range=(0, 10))
    texture_f = np.concatenate([hist_txt.astype(np.float32),
                                hist_lbp.astype(np.float32)])
    texture_f /= (np.linalg.norm(texture_f) + 1e-10)

    # --- SEKIL: kenar histogrami + Hu momentleri + aspect ratio ---
    edges = cv2.Canny(gray, 50, 150)
    hist_edge, _ = np.histogram(edges, bins=32, range=(0, 255))
    hu = cv2.HuMoments(cv2.moments(edges)).flatten()
    hu_log = -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)
    hu_log = np.nan_to_num(hu_log, nan=0.0, posinf=0.0, neginf=0.0)
    aspect_ratio = float(W_img) / float(H_img)
    shape_f = np.concatenate([hist_edge.astype(np.float32),
                              hu_log.astype(np.float32),
                              np.array([aspect_ratio], dtype=np.float32)])
    shape_f /= (np.linalg.norm(shape_f) + 1e-10)

    return color_f, shape_f, texture_f

# ---------- betimleyiciler (onbellekli, kesintiye dayanikli) ----------
if os.path.exists(WCV_PATH):
    with open(WCV_PATH, "rb") as f: WCV = pickle.load(f)
    print(f"Onbellek: {len(WCV)} gorsel")
else:
    WCV = {}

_kalan = [i for i in IMG_IDS if i not in WCV]
print(f"Uretilecek: {len(_kalan)}")
for k, iid in enumerate(tqdm(_kalan, desc="butun CV oznitelikleri")):
    idx = int(iid.split("_")[-1])
    c, s, t = _butun_betimleyici(ds_img[idx]["image"])
    WCV[iid] = {"color": c, "shape": s, "texture": t}
    if (k + 1) % 400 == 0:
        with open(WCV_PATH, "wb") as f: pickle.dump(WCV, f)
if _kalan:
    with open(WCV_PATH, "wb") as f: pickle.dump(WCV, f)

for m in MODLAR:
    _d = WCV[IMG_IDS[0]][m]
    print(f"  {m:<8} boyut {_d.shape[0]:>3}  norm {np.linalg.norm(_d):.3f}")

# ---------- mod kalibrasyonu: percentile-rank icin sirali ornek ----------
# Yama duzeyinde ayni-gorsel-ici rastgele yama ciftleri kullaniliyordu. Butun
# duzeyinde "gorsel ici" diye bir sey olmadigi icin karsiligi havuzdan rastgele
# iki AYRI gorseldir; aday havuzu da zaten baska gorsellerden olustugu icin
# referans dagilimi dogru olani budur.
def _butun_mod_kalibrasyonu(n_samples=3000, seed=42):
    rng = random.Random(seed)
    ids = sorted(IMG_IDS)
    stats = {}
    for m in MODLAR:
        sims, dene = [], 0
        while len(sims) < n_samples and dene < n_samples * 5:
            dene += 1
            a, b = rng.sample(ids, 2)
            fa, fb = WCV[a][m], WCV[b][m]
            na, nb = np.linalg.norm(fa), np.linalg.norm(fb)
            if na == 0 or nb == 0: continue
            sims.append(float(np.dot(fa, fb) / (na * nb)))
        stats[m] = np.sort(np.asarray(sims, dtype=np.float32))
    return stats

if os.path.exists(WCAL_PATH):
    with open(WCAL_PATH, "rb") as f: WMODE_CALIB = pickle.load(f)
    print(f"\nOnbellek: mod kalibrasyonu yuklendi")
else:
    print("\nMod kalibrasyonu hesaplaniyor (her mod icin 3000 rastgele gorsel cifti)...")
    WMODE_CALIB = _butun_mod_kalibrasyonu()
    with open(WCAL_PATH, "wb") as f: pickle.dump(WMODE_CALIB, f)

print("\nMod kalibrasyonu (rastgele gorsel ciftlerinin yuzey benzerligi dagilimi):")
for m, sv in WMODE_CALIB.items():
    print(f"  {m:<8}: ortalama={sv.mean():.4f}  std={sv.std():.4f}  N={len(sv)}")

def _yuzdelik(raw_sim, mode):
    """Yama duzeyindeki _calibrated ile ayni: ham benzerligin kalibrasyon
    dagilimindaki yuzdelik konumu [0,1]."""
    sv = WMODE_CALIB.get(mode)
    if sv is None or len(sv) == 0: return float(raw_sim)
    return float(np.searchsorted(sv, raw_sim) / len(sv))

# mod betimleyicilerini matrise al (hizli kosinus icin)
WCV_MAT = {m: np.stack([WCV[i][m] for i in IMG_IDS]).astype(np.float32) for m in MODLAR}
for m in MODLAR:
    WCV_MAT[m] /= (np.linalg.norm(WCV_MAT[m], axis=1, keepdims=True) + 1e-9)
print(f"\nhazir | betimleyici matrisleri: " +
      ", ".join(f"{m}{WCV_MAT[m].shape}" for m in MODLAR))

In [ ]:
# HÜCRE W4a: MultimodalSpiralEngine (yama duzeyinden, iki yapisal duzenleme ile)
# 1) img_shift govdesi _img_shift_sec metoduna tasindi -- davranis AYNI.
# 2) img_shift hedefi visited_image_ids'e de eklendi -- yama duzeyinde etkisiz.

class MultimodalSpiralEngine:
    def __init__(self, index: HybridMultimodalIndex, similarity_threshold: float = 0.55,
                 max_iou_shift: float = 0.1, verbose: bool = True, min_win_size: int = 2,
                 hardness_min: float = 0.05, hard_neg_sem_thresh: float = 0.85,
                 mode_calib: Optional[dict] = None,
                 allowed_image_ids: Optional[Set[str]] = None,
                 top_k: int = 1, diversity_alpha: float = 0.0):
        self.index = index
        self.threshold = similarity_threshold
        self.max_iou_shift = max_iou_shift
        self.verbose = verbose
        self.min_win_size = min_win_size
        self.hardness_min = hardness_min
        self.hard_neg_sem_thresh = hard_neg_sem_thresh
        self._mode_field = {"semantic": "embedding", "color": "color_feat", "shape": "shape_feat", "texture": "texture_feat"}
        # v5: modlar-arasi adil karsilastirma icin z-normalizasyon istatistikleri
        # (bkz. Bolum 3b -- shape_feat'in dogmus olmasi shape modunu yapay
        # olarak kayirmasin diye).
        self.mode_calib = mode_calib or {}  # v5b: bos birak -> _calibrated ham sim'i aynen dondurur (fallback)
        # v5d: TRAIN/TEST leakage duzeltmesi -- motor, txt_to_img adiminda
        # zincirin baslangicindan BAGIMSIZ olarak TUM index'ten en yakin
        # goruseli ariyordu (train/test ayrimini bilmiyordu). allowed_image_ids
        # verilirse, arama havuzu SADECE bu kumeye kisitlanir.
        self.allowed_image_ids = set(allowed_image_ids) if allowed_image_ids is not None else None
        if self.allowed_image_ids is not None:
            self._forbidden_image_ids = set(self.index.image_ids()) - self.allowed_image_ids
        else:
            self._forbidden_image_ids = set()
        # v5i: HUB-ETKISI DUZELTMESI -- query_text/query_image eskiden saf
        # argmax'ti (deterministik, top_k=1), yani birbirine yakin embedding'li
        # farkli sorgular hep AYNI hedefe yakinsiyordu, ve 600 sekans arasinda
        # HICBIR paylasilan hafiza olmadigi icin bu yakinsama sessizce
        # tekrarlaniyordu (train havuzunun sadece %43'u kullanilmisti).
        # Simdi: top_k aday arasindan, ASIRI KULLANILANLARI cezalandirarak
        # (kullanim_sayisi * diversity_alpha kadar skor dusurerek) seciyoruz --
        # bu sayac TUM run_hybrid_loop cagrilari boyunca (yani tum sekanslar
        # arasinda) KALICI, kendi kendini duzelten adaptif bir mekanizma.
        self.top_k = top_k
        self.diversity_alpha = diversity_alpha
        self.image_use_count = {}
        self.doc_use_count = {}

    def _get_feat(self, patch, mode: str):
        return getattr(patch, self._mode_field.get(mode, "embedding"), None)

    def _calibrated(self, raw_sim: float, mode: str) -> float:
        """v5b: percentile-rank normalizasyonu (z-skor DEGIL -- z-skor, std'si
        cok kucuk olan modlari (shape) yapay sekilde ezmisti/kazandirmisti).
        mode_calib[mode], calibrate_mode_stats'tan gelen SIRALI bir dizi;
        raw_sim'in bu dizideki yuzdelik konumunu [0,1] olarak dondururuz."""
        sorted_sims = self.mode_calib.get(mode)
        if sorted_sims is None or not hasattr(sorted_sims, "__len__") or len(sorted_sims) == 0:
            return raw_sim
        rank = np.searchsorted(sorted_sims, raw_sim) / len(sorted_sims)
        return float(rank)

    def _cosine_sim(self, a, b, fallback=0.0):
        if a is None or b is None:
            return fallback
        na, nb = np.linalg.norm(a), np.linalg.norm(b)
        if na == 0 or nb == 0:
            return fallback
        return float(np.dot(a, b) / (na * nb))

    def _pick_diverse(self, candidates, use_count, key_fn):
        """v5i: top-K adaylar arasindan, kullanim sayisina gore cezalandirilmis
        skora bakarak sec (asiri kullanilan -- hub -- adaylar geri plana
        duser). Secilenin kullanim sayacini 1 artirir. Dondurulen skor HAM
        (cezasiz) skordur -- loglama/rapor tutarliligi icin."""
        if self.diversity_alpha <= 0.0 or len(candidates) <= 1:
            best = candidates[0]
        else:
            best = max(candidates, key=lambda item: item[1] - self.diversity_alpha * use_count.get(key_fn(item[0]), 0))
        key = key_fn(best[0])
        use_count[key] = use_count.get(key, 0) + 1
        return best

    def _img_shift_sec(self, cur_patch, visited_patch_ids):
        """img_shift adiminin tamami: aday havuzu, uc yuzey modunun yarismasi,
        esik ve etiket. Govde HUCRE 4'ten BIREBIR tasindi -- tek fark, ayri bir
        metoda alinmis olmasi. Butun duzeyi alt sinifi SADECE bu metodu
        override eder, run_hybrid_loop'a hic dokunmaz.
        Dondurur: (tgt_img, mode, score, computed_label)"""
        MIN_OBJ = 0.4
        img_pool_raw = self.index.image_patches_of(cur_patch.image_id)
        valid_pool = [p for p in img_pool_raw if p.patch_id not in visited_patch_ids
                      and cur_patch.iou(p) <= self.max_iou_shift and p.objectness_score >= MIN_OBJ]
        if not valid_pool:
            valid_pool = sorted([p for p in img_pool_raw if p.patch_id not in visited_patch_ids],
                                 key=lambda x: x.objectness_score, reverse=True)[:5]

        if valid_pool:
            best_mode, best_patch = None, None
            best_surface_sim, best_sem_sim, best_hardness = 0.0, 0.0, -999.0
            for m in ("color", "shape", "texture"):
                q_feat = self._get_feat(cur_patch, m)
                if q_feat is None:
                    continue
                candidate = max(valid_pool, key=lambda p: self._cosine_sim(q_feat, self._get_feat(p, m), fallback=-1.0))
                surface_sim = self._cosine_sim(q_feat, self._get_feat(candidate, m), fallback=-1.0)
                sem_sim = self._cosine_sim(cur_patch.embedding, candidate.embedding)
                # v5: hardness karsilastirmasi z-normalize edilmis surface_sim ile yapiliyor
                # (raporlanan best_surface_sim yine HAM deger olarak kaydediliyor).
                surface_sim_z = self._calibrated(surface_sim, m)
                hardness = surface_sim_z - sem_sim
                if hardness > best_hardness:
                    best_mode, best_patch = m, candidate
                    best_surface_sim, best_sem_sim, best_hardness = surface_sim, sem_sim, hardness

            # NOT: best_hardness artik z-skor farkindan geldigi icin, hardness_min
            # esigi de v5'te yeniden kalibre edilmis olmali (asagidaki HUCRE E1'e bak).
            if best_patch is not None and best_hardness >= self.hardness_min:
                mode = best_mode
                tgt_img = best_patch
                score = best_surface_sim
                computed_label = "hard_negative" if best_sem_sim < self.hard_neg_sem_thresh else "positive"
            else:
                mode = "semantic"
                tgt_img = secrets.choice(valid_pool)
                score = self._cosine_sim(cur_patch.embedding, tgt_img.embedding)
                computed_label = "positive"
        else:
            mode = "semantic"
            score = 0.0
            computed_label = "positive"
            others = [p for p in img_pool_raw if p.patch_id != cur_patch.patch_id]
            tgt_img = min(others, key=lambda p: cur_patch.iou(p)) if others else cur_patch
        return tgt_img, mode, score, computed_label

    def run_hybrid_loop(self, start_image_id: Optional[str] = None, loops: int = 16) -> List[HybridStepRecord]:
        log: List[HybridStepRecord] = []
        all_image_ids = self.index.image_ids()
        if self.allowed_image_ids is not None:
            all_image_ids = [i for i in all_image_ids if i in self.allowed_image_ids]
        if not all_image_ids:
            return log

        if start_image_id is None or start_image_id not in all_image_ids:
            start_image_id = secrets.choice(all_image_ids)

        visited_image_ids: Set[str] = {start_image_id}
        visited_doc_ids:   Set[str] = set()
        visited_patch_ids: Set[str] = set()

        pool = self.index.image_patches_of(start_image_id)
        obj_pool = [p for p in pool if p.type == "object"]
        cur_patch = secrets.choice(obj_pool if obj_pool else pool)
        visited_patch_ids.add(cur_patch.patch_id)

        log.append(HybridStepRecord(step=0, action="start", modality="image",
                                     image_id=start_image_id, patch_id=cur_patch.patch_id,
                                     bbox=cur_patch.bbox, similarity_mode="semantic"))
        step_n = 1

        res = self.index.query_text(cur_patch.embedding, exclude_doc_ids=visited_doc_ids,
                                     min_win_size=self.min_win_size, threshold=self.threshold, top_k=self.top_k)
        if not res:
            res = self.index.query_text(cur_patch.embedding, exclude_doc_ids=visited_doc_ids,
                                         min_win_size=self.min_win_size, threshold=-1.0, top_k=self.top_k)
        if not res:
            return log

        tgt_txt, sim0 = self._pick_diverse(res, self.doc_use_count, lambda p: p.doc_id)
        visited_doc_ids.add(tgt_txt.doc_id)
        visited_patch_ids.add(tgt_txt.patch_id)
        hn_txt = self.index.find_hard_negative_text(tgt_txt, visited_doc_ids,
                                                      hardness_min=self.hardness_min,
                                                      hard_neg_sem_thresh=self.hard_neg_sem_thresh)
        log.append(HybridStepRecord(step=step_n, action="img_to_txt", modality="text",
                                     doc_id=tgt_txt.doc_id, patch_id=tgt_txt.patch_id,
                                     text_content=tgt_txt.text_content, similarity=sim0, similarity_mode="semantic",
                                     hard_negative_txt_patch_id=hn_txt[0] if hn_txt else None,
                                     hard_negative_txt_sim=hn_txt[1] if hn_txt else None,
                                     hard_negative_txt_mode=hn_txt[2] if hn_txt else None))
        step_n += 1

        for lp in range(1, loops + 1):
            txt_pool = [p for p in self.index.text_patches_of(tgt_txt.doc_id)
                        if p.patch_id not in visited_patch_ids and p.win_size >= self.min_win_size
                        and self.index._is_quality_text(p.text_content)]
            if not txt_pool:
                txt_pool = [p for p in self.index.text_patches_of(tgt_txt.doc_id) if p.patch_id not in visited_patch_ids]
            if txt_pool:
                tgt_txt = max(txt_pool, key=lambda p: self._cosine_sim(tgt_txt.embedding, p.embedding))
                visited_patch_ids.add(tgt_txt.patch_id)
                log.append(HybridStepRecord(step=step_n, action="txt_shift", modality="text",
                                             doc_id=tgt_txt.doc_id, patch_id=tgt_txt.patch_id,
                                             text_content=tgt_txt.text_content, similarity_mode="semantic"))
                step_n += 1

            # v5d: ziyaret edilenlere EK olarak, izin verilen havuzun DISINDAKI
            # tum gorselleri de haric tut -- boylece train sekansi test
            # gorseline (ya da tam tersi) SICRAYAMAZ.
            img_res = self.index.query_image(tgt_txt.embedding, exclude_image_ids=visited_image_ids | self._forbidden_image_ids, top_k=self.top_k)
            if not img_res:
                break
            best_img, best_score = self._pick_diverse(img_res, self.image_use_count, lambda p: p.image_id)
            visited_image_ids.add(best_img.image_id)
            visited_patch_ids.add(best_img.patch_id)
            log.append(HybridStepRecord(step=step_n, action="txt_to_img", modality="image",
                                         image_id=best_img.image_id, patch_id=best_img.patch_id,
                                         bbox=best_img.bbox, similarity=best_score, similarity_mode="semantic"))
            step_n += 1
            cur_patch = best_img

            tgt_img, mode, score, computed_label = self._img_shift_sec(
                cur_patch, visited_patch_ids)
            if tgt_img is None:
                break

            visited_patch_ids.add(tgt_img.patch_id)
            # Yama duzeyinde img_shift ayni sahnede kaldigi icin bu satir
            # ETKISIZDIR (gorsel zaten visited_image_ids'te). Butun duzeyinde
            # img_shift baska bir gorsele gectigi icin gereklidir.
            visited_image_ids.add(tgt_img.image_id)
            log.append(HybridStepRecord(step=step_n, action="img_shift", modality="image",
                                         image_id=tgt_img.image_id, patch_id=tgt_img.patch_id,
                                         bbox=tgt_img.bbox, similarity=score,
                                         similarity_mode=mode, verified_label=computed_label))
            step_n += 1
            cur_patch = tgt_img

            res = self.index.query_text(cur_patch.embedding, exclude_doc_ids=visited_doc_ids,
                                         min_win_size=self.min_win_size, threshold=self.threshold, top_k=self.top_k)
            if not res:
                res = self.index.query_text(cur_patch.embedding, exclude_doc_ids=visited_doc_ids,
                                             min_win_size=self.min_win_size, threshold=-1.0, top_k=self.top_k)
            if not res:
                break
            tgt_txt, sim = self._pick_diverse(res, self.doc_use_count, lambda p: p.doc_id)
            visited_doc_ids.add(tgt_txt.doc_id)
            visited_patch_ids.add(tgt_txt.patch_id)
            hn_txt = self.index.find_hard_negative_text(tgt_txt, visited_doc_ids,
                                                          hardness_min=self.hardness_min,
                                                          hard_neg_sem_thresh=self.hard_neg_sem_thresh)
            log.append(HybridStepRecord(step=step_n, action="img_to_txt", modality="text",
                                         doc_id=tgt_txt.doc_id, patch_id=tgt_txt.patch_id,
                                         text_content=tgt_txt.text_content, similarity=sim, similarity_mode="semantic",
                                         hard_negative_txt_patch_id=hn_txt[0] if hn_txt else None,
                                         hard_negative_txt_sim=hn_txt[1] if hn_txt else None,
                                         hard_negative_txt_mode=hn_txt[2] if hn_txt else None))
            step_n += 1

        return log

In [ ]:
# HÜCRE W4b: butun duzeyi motoru + butun-gorsel indeksi
# ---------------------------------------------------------------------------
# Motorun TEK farki img_shift adiminin aday havuzudur. Yama duzeyinde bu adim
# AYNI SAHNENIN baska bir yamasina gecer; bir butun gorselin "baska bir
# parcasi" olmadigi icin butun duzeyinde adaylar havuzdaki diger butun
# gorsellerdir. Yarisma kurali, kalibrasyon, h_m = yuzdelik(s_m) - s_sem,
# tau_h ve tau_s esikleri ve etiketleme aynen korunur.
#
# Ikinci fark: yama duzeyindeki secrets.choice yerine tohumlanmis rng
# kullanilir, boylece zincirler yeniden uretilebilir olur.
import random
import numpy as np

class ButunSarmalMotor(MultimodalSpiralEngine):
    def __init__(self, *a, rng_seed=1234, aday_havuz=None, **kw):
        super().__init__(*a, **kw)
        self._rng = random.Random(rng_seed)
        # aday_havuz: img_shift adiminda yarisacak aday sayisi.
        #   None -> havuzdaki TUM uygun gorseller
        #   N    -> anlamsal olarak en yakin N gorsel
        # Yama duzeyinde bu havuz "ayni sahnenin diger yamalari"dir, yani hem
        # capa ile ILISKILI hem de KUCUKTUR. 1275 gorselin tamami alinirsa
        # yuzey benzerliginin maksimumu kalibrasyon dagiliminin tepesine
        # oturur, yuzdelik 1'e yakinsar ve h >= tau_h kosulu neredeyse her
        # adimda saglanir; mod yarismasi ayirt edici olmaktan cikar.
        self.aday_havuz = aday_havuz
        P = self.index.image_patches
        self._pid_kod = {p.patch_id: k for k, p in enumerate(P)}
        self._iid_kod = {p.image_id: k for k, p in enumerate(P)}
        assert len(self._iid_kod) == len(P), \
            "butun indeksinde gorsel basina tam 1 dugum olmali"
        self._sem = np.stack([p.embedding for p in P]).astype(np.float32)
        self._mod_mat = {}
        for m, alan in (("color", "color_feat"), ("shape", "shape_feat"),
                        ("texture", "texture_feat")):
            M = np.stack([getattr(p, alan) for p in P]).astype(np.float32)
            M /= (np.linalg.norm(M, axis=1, keepdims=True) + 1e-9)
            self._mod_mat[m] = M
        self._izinli = np.array(
            [self.allowed_image_ids is None or p.image_id in self.allowed_image_ids
             for p in P], dtype=bool)

    def _img_shift_sec(self, cur_patch, visited_patch_ids):
        """Yama duzeyindeki _img_shift_sec ile AYNI karar zinciri; aday havuzu
        havuzdaki diger butun gorseller ve hesap vektorlestirilmis."""
        q = self._iid_kod[cur_patch.image_id]
        uygun = self._izinli.copy()
        uygun[q] = False
        for pid in visited_patch_ids:
            k = self._pid_kod.get(pid)
            if k is not None:
                uygun[k] = False
        idx = np.flatnonzero(uygun)
        if idx.size == 0:
            return None, "semantic", 0.0, "positive"
        if self.aday_havuz is not None and idx.size > self.aday_havuz:
            _s = self._sem[idx] @ self._sem[q]
            _sec = np.argpartition(-_s, self.aday_havuz - 1)[:self.aday_havuz]
            idx = idx[_sec]

        sem = self._sem[idx] @ self._sem[q]

        # --- uc yuzey modu yarisir (yama duzeyiyle ayni kural) ---
        best_mode, best_j = None, None
        best_surface_sim, best_sem_sim, best_hardness = 0.0, 0.0, -999.0
        for m in ("color", "shape", "texture"):
            M = self._mod_mat[m]
            s = M[idx] @ M[q]
            j = int(np.argmax(s))
            surface_sim = float(s[j])
            sem_sim = float(sem[j])
            hardness = self._calibrated(surface_sim, m) - sem_sim
            if hardness > best_hardness:
                best_mode, best_j = m, j
                best_surface_sim, best_sem_sim, best_hardness = surface_sim, sem_sim, hardness

        if best_j is not None and best_hardness >= self.hardness_min:
            tgt_img = self.index.image_patches[int(idx[best_j])]
            label = "hard_negative" if best_sem_sim < self.hard_neg_sem_thresh else "positive"
            return tgt_img, best_mode, best_surface_sim, label

        j = self._rng.randrange(int(idx.size))          # yama duzeyindeki secrets.choice karsiligi
        tgt_img = self.index.image_patches[int(idx[j])]
        return tgt_img, "semantic", float(sem[j]), "positive"


# ---------------------------------------------------------------------------
# butun-gorsel indeksi: gorsel basina TAM 1 dugum (bbox tum kare), + ayni
# hikaye cumlesi pencereleri ve ayni TF-IDF leksik oznitelikleri
# ---------------------------------------------------------------------------
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer

butun_index = HybridMultimodalIndex()

print("1/2: metin yamalari indekse ekleniyor...")
with open(f"{DRIVE_BASE}/ZoomInOut/simple_story_sentences_embeddings_new_tr_5000.pkl", "rb") as f:
    all_text_metadata = pickle.load(f)
_tfidf = TfidfVectorizer(max_features=512, min_df=2)
_lex = _tfidf.fit_transform([m["text_content"] for m in all_text_metadata]).toarray().astype(np.float32)
for i, t in enumerate(all_text_metadata):
    butun_index.add_text_patch(TextPatchMeta(
        doc_id=t["doc_id"], patch_id=t["patch_id"],
        text_content=t["text_content"], embedding=t["embedding"],
        win_size=int(t.get("win_size", 2)), win_label=t.get("win_label", "w2"),
        richness_score=float(t.get("richness_score", 0.0)),
        lexical_feat=_lex[i]))

print("2/2: butun gorseller indekse ekleniyor...")
for iid in IMG_IDS:
    butun_index.add_image_patch(ImagePatchMeta(
        image_id=iid, patch_id=f"{iid}_whole", bbox=[0, 0, 224, 224],
        embedding=WHOLE[iid], objectness_score=1.0, type="object",
        color_feat=WCV[iid]["color"], shape_feat=WCV[iid]["shape"],
        texture_feat=WCV[iid]["texture"]))

print(f"\nIndeks hazir: {len(butun_index.image_patches)} butun gorsel, "
      f"{len(butun_index.text_patches)} metin yamasi")
assert len(butun_index.image_ids()) == len(IMG_IDS)

# ---------------------------------------------------------------------------
# yama duzeyiyle AYNI hiperparametreler
# ---------------------------------------------------------------------------
SIM_THRESHOLD   = 0.55
NUM_SEQUENCES   = 600
LOOPS_PER_SEQ   = 16
MIN_CHAIN_LEN   = 16
TOP_K           = 8
DIVERSITY_ALPHA = 0.03
MIN_WIN_SIZE    = 2
HARDNESS_MIN    = 0.1     # tau_h
HARD_NEG_SEM_TH = 0.85    # tau_s
TEST_NUM_SEQ, TEST_TARGET = 600, 400
VAL_NUM_SEQ,  VAL_TARGET  = 300, 150

# img_shift aday havuzunun boyutu, yama duzeyindeki havuzun OLCULEN
# ortalamasina esitlenir: ayni gorselin objectness >= 0.4 olan yama sayisi.
from collections import Counter as _Counter
_pps = _Counter(m["image_id"] for m in layout
                if m.get("objectness_score", 1.0) >= 0.4)
ADAY_HAVUZ = max(2, int(round(float(np.mean(list(_pps.values()))))))
print(f"\nyama duzeyinde img_shift aday havuzu ortalamasi: "
      f"{np.mean(list(_pps.values())):.1f} yama/gorsel -> ADAY_HAVUZ={ADAY_HAVUZ}")

def motor_kur(izinli_ids, rng_seed=1234, aday_havuz=ADAY_HAVUZ):
    return ButunSarmalMotor(
        butun_index, similarity_threshold=SIM_THRESHOLD, verbose=False,
        min_win_size=MIN_WIN_SIZE, hardness_min=HARDNESS_MIN,
        hard_neg_sem_thresh=HARD_NEG_SEM_TH, mode_calib=WMODE_CALIB,
        allowed_image_ids=izinli_ids, top_k=TOP_K,
        diversity_alpha=DIVERSITY_ALPHA, rng_seed=rng_seed,
        aday_havuz=aday_havuz)

print(f"motor hazir | tau_h={HARDNESS_MIN} tau_s={HARD_NEG_SEM_TH} "
      f"top_k={TOP_K} alpha={DIVERSITY_ALPHA} esik={SIM_THRESHOLD} "
      f"aday_havuz={ADAY_HAVUZ}")

In [ ]:
# HÜCRE W4b2: hizli indeks -- ayni kararlar, vektorlestirilmis
# ---------------------------------------------------------------------------
# query_text, query_image ve find_hard_negative_text her cagrida 5000 metin
# yamasi uzerinde Python dongusu kuruyor ve her yamada _is_quality_text
# cagirarak text.split() yapiyordu. Sekans basina yaklasik 255 bin string
# islemi, yani ~21 saniye. Kalite ve win_size filtreleri SABIT oldugu icin
# bir kez onceden hesaplanabilir, dis dongu de numpy'a tasinabilir.
#
# Kararlar birebir ayni: ayni agirliklar, ayni -999 maskesi, ayni argmax /
# argsort cagrilari, ayni esik karsilastirmalari. Sadece hiz degisir.
import numpy as np

class HizliIndex(HybridMultimodalIndex):
    def _hazirla(self):
        self._build_matrices()
        T = self.text_patches
        self._t_doc = np.array([p.doc_id for p in T])
        self._t_win = np.array([p.win_size for p in T], dtype=np.int32)
        self._t_kalite = np.array([self._is_quality_text(p.text_content) for p in T])
        _WW = {1: 0.70, 2: 1.05, 3: 1.15}
        self._t_wagirlik = np.array([_WW.get(int(w), 1.0) for w in self._t_win],
                                    dtype=np.float32)
        I = self.image_patches
        self._i_img = np.array([p.image_id for p in I])
        _TW = {"object": 1.10, "background": 0.92}
        self._i_tagirlik = np.array([_TW.get(p.type, 1.0) for p in I], dtype=np.float32)
        print(f"hizli indeks hazir | kaliteli metin yamasi "
              f"{int(self._t_kalite.sum())}/{len(T)}")

    def query_text(self, query_emb, exclude_doc_ids=None, threshold=-1.0,
                   min_win_size=1, top_k=1):
        exclude_doc_ids = exclude_doc_ids or set()
        q = query_emb / (np.linalg.norm(query_emb) + 1e-9)
        raw = np.clip(self._txt_matrix @ q, -1.0, 1.0)
        uygun = self._t_kalite & (self._t_win >= min_win_size)
        if exclude_doc_ids:
            uygun &= ~np.isin(self._t_doc, np.array(sorted(exclude_doc_ids)))
        weighted = raw * self._t_wagirlik
        weighted[~uygun] = -999.0
        if top_k <= 1:
            b = int(np.argmax(weighted))
            if weighted[b] <= -999.0: return []
            return [(self.text_patches[b], float(raw[b]))]
        order = np.argsort(-weighted)[:top_k]
        return [(self.text_patches[i], float(raw[i])) for i in order
                if weighted[i] > -999.0]

    def query_image(self, query_emb, exclude_image_ids=None, top_k=1):
        exclude_image_ids = exclude_image_ids or set()
        q = query_emb / (np.linalg.norm(query_emb) + 1e-9)
        raw = np.clip(self._img_matrix @ q, -1.0, 1.0)
        weighted = raw * self._i_tagirlik
        if exclude_image_ids:
            weighted[np.isin(self._i_img, np.array(sorted(exclude_image_ids)))] = -999.0
        if top_k <= 1:
            b = int(np.argmax(weighted))
            if weighted[b] <= -999.0: return []
            return [(self.image_patches[b], float(raw[b]))]
        order = np.argsort(-weighted)[:top_k]
        return [(self.image_patches[i], float(raw[i])) for i in order
                if weighted[i] > -999.0]

    def find_hard_negative_text(self, anchor_patch, exclude_doc_ids,
                                hardness_min=0.05, hard_neg_sem_thresh=0.85):
        if self._lex_matrix is None or anchor_patch.lexical_feat is None:
            return None
        surface_all = np.clip(self._lex_matrix @ anchor_patch.lexical_feat, -1.0, 1.0)
        sem_all = np.clip(self._txt_matrix @ anchor_patch.embedding, -1.0, 1.0)
        hardness_all = surface_all - sem_all
        uygun = self._t_kalite & (self._t_doc != anchor_patch.doc_id)
        if exclude_doc_ids:
            uygun &= ~np.isin(self._t_doc, np.array(sorted(exclude_doc_ids)))
        if not uygun.any(): return None
        h = np.where(uygun, hardness_all, -np.inf)
        b = int(np.argmax(h))
        if h[b] < hardness_min or sem_all[b] >= hard_neg_sem_thresh:
            return None
        return self.text_patches[b].patch_id, float(surface_all[b]), "lexical"

# mevcut indeksi yerinde hizlandir (yeni nesne kurulmaz, motor_kur degismez)
butun_index.__class__ = HizliIndex
butun_index._hazirla()

# --- esdegerlik kontrolu: yavas ve hizli yol ayni sonucu vermeli ---
import random as _r
_ref = HybridMultimodalIndex.__dict__
_rng = _r.Random(0)
_tp = butun_index.text_patches; _ip = butun_index.image_patches
_hata = 0
for _ in range(25):
    _q = _ip[_rng.randrange(len(_ip))].embedding
    _ex = {_tp[_rng.randrange(len(_tp))].doc_id for _ in range(3)}
    a = _ref["query_text"](butun_index, _q, exclude_doc_ids=_ex, min_win_size=2, top_k=8)
    b = butun_index.query_text(_q, exclude_doc_ids=_ex, min_win_size=2, top_k=8)
    if [p.patch_id for p, _ in a] != [p.patch_id for p, _ in b]: _hata += 1
    _qt = _tp[_rng.randrange(len(_tp))]
    a2 = _ref["find_hard_negative_text"](butun_index, _qt, _ex, 0.05, 0.85)
    b2 = butun_index.find_hard_negative_text(_qt, _ex, 0.05, 0.85)
    if (a2 or (None,))[0] != (b2 or (None,))[0]: _hata += 1
    _ex2 = {_ip[_rng.randrange(len(_ip))].image_id for _ in range(5)}
    a3 = _ref["query_image"](butun_index, _q, exclude_image_ids=_ex2, top_k=8)
    b3 = butun_index.query_image(_q, exclude_image_ids=_ex2, top_k=8)
    if [p.patch_id for p, _ in a3] != [p.patch_id for p, _ in b3]: _hata += 1
print(f"esdegerlik kontrolu: {75 - _hata}/75 sorgu ayni" +
      ("  -- TAMAM" if _hata == 0 else f"  -- {_hata} FARK VAR, DURDUR"))
assert _hata == 0, "hizli indeks yavas yoldan farkli sonuc veriyor"

In [ ]:
# HÜCRE W4c: tau_h kalibrasyonu -- yama duzeyindeki HÜCRE E1'in karsiligi
# ---------------------------------------------------------------------------
# Yama duzeyinde tau_h = 0.1 bir sabit DEGIL, kalibre edilmis bir degerdir:
# HÜCRE E1 esikleri tarayip %15-30 verim veren ve mod dagilimi dengeli olan
# degeri secmekte, HÜCRE 18 de oznitelik olcegi degisince eski esigin
# anlamsiz kalacagini belirtmektedir.
#
# Butun gorsellerde olcek gercekten degismistir. W3'un kalibrasyon ciktisi:
#   color   ortalama 0.5938  std 0.1797
#   shape   ortalama 0.9898  std 0.0153
#   texture ortalama 0.8775  std 0.1037
# Sekil betimleyicisi tam fotograflarda neredeyse sabittir; yuzdelik-rank bu
# yuzden neredeyse her adayi tepeye tasir ve h ~ 1 - s_sem olur. tau_h = 0.1
# bu olcekte her adimda saglanir. Esik burada yeniden kalibre edilir --
# yontem ayni, kalibrasyon adimi da yontemin parcasi.
#
# E1 gibi her esikte zincirler BASTAN uretilir: esik sadece etiketi degil,
# zincirin bir sonraki dugumunu de belirledigi icin tarama baska turlu
# dogru olmaz.
from collections import Counter
import numpy as np, time

KALIB_SEKANS = 30
TAU_H_TARAMA = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]
HEDEF_ALT, HEDEF_UST = 15.0, 30.0      # HÜCRE E1'deki kabul araligi
YAMA_REF = 17.4                        # yama duzeyinde olculen verim

print(f"{'tau_h':<8}{'hard negative':>16}{'verim':>9}{'ort s_sem':>11}"
      f"{'renk':>7}{'sekil':>7}{'doku':>7}")
print("-" * 65)
_onerilen, _t0 = None, time.time()
for _th in TAU_H_TARAMA:
    _m = motor_kur(train_ids, rng_seed=1234, aday_havuz=ADAY_HAVUZ)
    _m.hardness_min = _th
    _shift, _hn, _sem, _md = 0, 0, [], Counter()
    for _s0 in sorted(train_ids)[:KALIB_SEKANS]:
        _c = _m.run_hybrid_loop(start_image_id=_s0, loops=LOOPS_PER_SEQ)
        _onceki = None
        for _s in _c:
            if _s.modality == "image" and _s.action != "img_shift":
                _onceki = _s.image_id
            elif _s.action == "img_shift":
                _shift += 1
                if _s.verified_label == "hard_negative":
                    _hn += 1; _md[_s.similarity_mode] += 1
                    if _onceki:
                        _a, _b = _m._iid_kod[_onceki], _m._iid_kod[_s.image_id]
                        _sem.append(float(_m._sem[_a] @ _m._sem[_b]))
                _onceki = _s.image_id
    _v = _hn / max(_shift, 1) * 100
    _isaret = ""
    if HEDEF_ALT <= _v <= HEDEF_UST:
        _isaret = " <-- kabul araliginda"
        if _onerilen is None or abs(_v - YAMA_REF) < _onerilen[1]:
            _onerilen = (_th, abs(_v - YAMA_REF), _v)
    print(f"{_th:<8}{_hn:>8}/{_shift:<7}{_v:>8.1f}%"
          f"{(np.mean(_sem) if _sem else float('nan')):>11.3f}"
          f"{_md.get('color',0):>7}{_md.get('shape',0):>7}{_md.get('texture',0):>7}"
          f"{_isaret}")
print("-" * 65)
print(f"tarama {(time.time()-_t0)/60:.1f} dk | aday havuzu {ADAY_HAVUZ} | "
      f"tau_s {HARD_NEG_SEM_TH}")
print(f"kabul araligi %{HEDEF_ALT:.0f}-{HEDEF_UST:.0f} (HÜCRE E1 olcutu) | "
      f"yama duzeyi referansi %{YAMA_REF}")

# ---------------------------------------------------------------------------
# SECILEN ESIK
# ---------------------------------------------------------------------------
if _onerilen:
    HARDNESS_MIN = _onerilen[0]
    print(f"\nHARDNESS_MIN = {HARDNESS_MIN} (verim %{_onerilen[2]:.1f}, yama "
          f"duzeyine en yakin)")
else:
    print("\nHicbir esik kabul araliginda degil -- TAU_H_TARAMA araligini genislet.")
print(f"tau_s = {HARD_NEG_SEM_TH} degismedi: s_sem ayni kosinus olceginde.")
print("Mod dagilimi dengesizse veya baska bir deger kullanmak istersen "
      "HARDNESS_MIN'i burada elle yaz.")
print(f"\nEsigi degistirdikten sonra W5 onbellegini sil:\n  "
      f"{OUT}/butun_birebir_pairs.pkl")

In [ ]:
# HÜCRE W4d: cift cikarma fonksiyonlari + tau_h ince tarama
# ---------------------------------------------------------------------------
# generate_sequences, build_pairs_from_sequences ve attach_embeddings yama
# duzeyindeki HUCRE 6'dan alindi. TEK zorunlu uyarlama build_pairs icindeki
# "shift_step.image_id == img_step.image_id" kosuludur: yama duzeyinde
# img_shift ayni sahnede kaldigi icin bu esitlik dogrudur, butun duzeyinde
# img_shift tanim geregi baska bir gorsele gectigi icin esitlik degil
# FARKLILIK beklenir. Kosul oldugu gibi birakilsa butun duzeyinde hicbir hard
# negative iliştirilemezdi.
import random, pickle, os, time
import numpy as np
from tqdm.auto import tqdm

def sekans_uret(engine, image_ids, num_sequences, loops, min_chain_length, etiket):
    image_ids = list(image_ids)
    _rng = random.Random(1234)
    shuffled = image_ids[:]
    _rng.shuffle(shuffled)
    seqs = []
    for i in tqdm(range(num_sequences), desc=etiket):
        chain = engine.run_hybrid_loop(start_image_id=shuffled[i % len(shuffled)], loops=loops)
        if len(chain) >= min_chain_length:
            seqs.append(chain)
    print(f"  {len(seqs)}/{num_sequences} sekans (min_chain_length={min_chain_length})")
    return seqs

def ciftleri_cikar(sequences, sim_threshold):
    records, n_hn = [], 0
    for chain in sequences:
        for i, step in enumerate(chain):
            if step.action not in ("img_to_txt", "txt_to_img") or i == 0:
                continue
            prev = chain[i - 1]
            img_step, txt_step = (prev, step) if step.action == "img_to_txt" else (step, prev)
            if img_step.modality != "image" or txt_step.modality != "text":
                continue
            cross_sim = step.similarity if step.similarity else 0.0
            if cross_sim < sim_threshold:
                continue
            record = {
                "txt_patch_id": txt_step.patch_id, "img_patch_id": img_step.patch_id,
                "cross_modal_sim": cross_sim,
                "hard_negative_patch_id": None, "hard_negative_sim": None,
                "hard_negative_mode": None,
                "hard_negative_txt_patch_id": getattr(txt_step, "hard_negative_txt_patch_id", None),
                "hard_negative_txt_sim": getattr(txt_step, "hard_negative_txt_sim", None),
                "hard_negative_txt_mode": getattr(txt_step, "hard_negative_txt_mode", None),
            }
            if i + 1 < len(chain) and chain[i + 1].action == "img_shift":
                shift_step = chain[i + 1]
                # --- butun duzeyi uyarlamasi: == yerine != ---
                if (shift_step.image_id != img_step.image_id
                        and getattr(shift_step, "verified_label", "positive") == "hard_negative"):
                    record["hard_negative_patch_id"] = shift_step.patch_id
                    record["hard_negative_sim"] = shift_step.similarity
                    record["hard_negative_mode"] = shift_step.similarity_mode
                    n_hn += 1
            records.append(record)
    print(f"  cross-modal cift {len(records)} | hard negative'li {n_hn} "
          f"(%{n_hn/max(len(records),1)*100:.1f})")
    return records

def gommeleri_ekle(records, index, rng_seed=7):
    """Pozitif metin, yama duzeyindeki gibi gorselin KENDI COCO caption'i ile
    degistirilir; hikaye cumleleri yalnizca gezinmeyi yonlendirir."""
    rng = random.Random(rng_seed)
    txt_lookup = {p.patch_id: p.embedding for p in index.text_patches}
    img_lookup = {p.patch_id: p.embedding for p in index.image_patches}
    img_id_lookup = {p.patch_id: p.image_id for p in index.image_patches}
    final, n_cap = [], 0
    for r in records:
        txt_emb = txt_lookup.get(r["txt_patch_id"])
        img_emb = img_lookup.get(r["img_patch_id"])
        if txt_emb is None or img_emb is None:
            continue
        iid = img_id_lookup.get(r["img_patch_id"])
        coco_id = IID2COCO.get(iid)
        if coco_id is not None and coco_id in CAP_EMB:
            txt_emb = rng.choice(CAP_EMB[coco_id])[1]
            n_cap += 1
        else:
            continue                      # caption'i olmayan gorsel disarida kalir
        hn_emb = img_lookup.get(r["hard_negative_patch_id"]) \
                 if r["hard_negative_patch_id"] is not None else None
        final.append({
            "txt_embedding": np.asarray(txt_emb, dtype=np.float32),
            "img_embedding": np.asarray(img_emb, dtype=np.float32),
            "cross_modal_sim": r["cross_modal_sim"],
            "hard_negative_embedding": np.asarray(hn_emb, dtype=np.float32)
                                       if hn_emb is not None else None,
            "hard_negative_sim": r["hard_negative_sim"],
            "hard_negative_mode": r["hard_negative_mode"],
            "image_id": iid,
        })
    print(f"  final kayit {len(final)} | COCO caption kullanilan {n_cap}")
    return final

def degerlendirme_uret(engine, index, image_ids, num_seq, loops, min_len,
                        sim_threshold, target, etiket, rng_seed=11):
    """Yama duzeyindeki generate_test_pairs ile ayni: gorsel basina EN FAZLA
    1 ornek, hard negative filtresi YOK (train ile ayni dagilim)."""
    seqs = sekans_uret(engine, image_ids, num_seq, loops, min_len, etiket)
    recs = ciftleri_cikar(seqs, sim_threshold)
    random.Random(rng_seed).shuffle(recs)
    img_id_lookup = {p.patch_id: p.image_id for p in index.image_patches}
    gorulen, dedup = set(), []
    for r in recs:
        iid = img_id_lookup.get(r["img_patch_id"])
        if iid in gorulen:
            continue
        gorulen.add(iid); dedup.append(r)
        if len(dedup) >= target:
            break
    if len(dedup) < target:
        print(f"  UYARI: hedef {target}, bulunan {len(dedup)} benzersiz gorsel")
    return gommeleri_ekle(dedup, index, rng_seed=rng_seed)



def _sessiz_cikar(sequences, sim_threshold):
    """ciftleri_cikar ile AYNI, sadece cikti basmaz (tarama icinde kullanilir)."""
    import io as _io, contextlib as _ct
    with _ct.redirect_stdout(_io.StringIO()):
        return ciftleri_cikar(sequences, sim_threshold)


# ---------------------------------------------------------------------------
# tau_h ince tarama: adim duzeyi DEGIL, CIFT duzeyi oranina gore
# ---------------------------------------------------------------------------
# W4c, HÜCRE E1'in olcutunu kullanir ve verimi img_shift ADIMLARI uzerinden
# hesaplar. Egitimi belirleyen sey ise CIFT duzeyindeki orandir: yama
# duzeyinde egitim ciftlerinin %17.4'u bir hard negative tasir ve mod
# yardimci kaybi yalnizca o ciftlerde devreye girer.
#
# Zincir yapisi geregi ciftlerin yalnizca yarisi bir img_shift tarafindan
# izlenir (her dongude bir txt_to_img ve bir img_to_txt cifti olusur, ama tek
# bir img_shift vardir), dolayisiyla cift duzeyi orani adim duzeyi veriminin
# kabaca yarisidir. Bu tarama iki orani birlikte olcer, boylece yama
# duzeyindeki %17.4 ile eslesen esik secilebilir.
import time
from collections import Counter

INCE_TARAMA = [0.20, 0.21, 0.22, 0.23, 0.24, 0.25]
INCE_SEKANS = 30
YAMA_CIFT_REF = 17.4          # yama duzeyinde CIFT duzeyi hard negative orani

print(f"{'tau_h':<8}{'adim':>9}{'cift':>9}{'n cift':>9}{'renk':>7}{'sekil':>7}{'doku':>7}")
print("-" * 56)
_t0, _en = time.time(), None
for _th in INCE_TARAMA:
    _m = motor_kur(train_ids, rng_seed=1234, aday_havuz=ADAY_HAVUZ)
    _m.hardness_min = _th
    _sq = [_m.run_hybrid_loop(start_image_id=s, loops=LOOPS_PER_SEQ)
           for s in sorted(train_ids)[:INCE_SEKANS]]
    _shift = sum(1 for c in _sq for s in c if s.action == "img_shift")
    _hn_s = sum(1 for c in _sq for s in c
                if s.action == "img_shift" and s.verified_label == "hard_negative")
    _rec = [r for r in _sessiz_cikar(_sq, SIM_THRESHOLD)]
    _hn_c = sum(1 for r in _rec if r["hard_negative_patch_id"] is not None)
    _o_s = _hn_s / max(_shift, 1) * 100
    _o_c = _hn_c / max(len(_rec), 1) * 100
    _md = Counter(r["hard_negative_mode"] for r in _rec
                  if r["hard_negative_mode"])
    _is = ""
    if _en is None or abs(_o_c - YAMA_CIFT_REF) < _en[1]:
        _en = (_th, abs(_o_c - YAMA_CIFT_REF), _o_c, _o_s)
    print(f"{_th:<8}{_o_s:>8.1f}%{_o_c:>8.1f}%{len(_rec):>9}"
          f"{_md.get('color',0):>7}{_md.get('shape',0):>7}{_md.get('texture',0):>7}")
print("-" * 56)
print(f"tarama {(time.time()-_t0)/60:.1f} dk | adim = img_shift adimlarinin orani, "
      f"cift = egitim ciftlerinin orani")
print(f"yama duzeyi referanslari: adim %15-30 (E1 araligi), cift %{YAMA_CIFT_REF}")
if _en:
    print(f"\nCIFT duzeyine gore en yakin: tau_h = {_en[0]} "
          f"(cift %{_en[2]:.1f}, adim %{_en[3]:.1f})")
print(f"\nSecilen deger: HARDNESS_MIN = {HARDNESS_MIN}")
print("Degistirmek istersen bu hucrenin sonunda HARDNESS_MIN'i elle yaz,")
print("sonra W5'i calistir.")

In [ ]:
# HÜCRE W5: sekans uretimi (fonksiyonlar W4d'de tanimli)
import random, pickle, os, time
import numpy as np
from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
W_PAIRS_PATH = f"{OUT}/butun_birebir_pairs.pkl"
if os.path.exists(W_PAIRS_PATH):
    with open(W_PAIRS_PATH, "rb") as f: _D = pickle.load(f)
    w_train_pairs, w_val_pairs, w_test_pairs = _D["train"], _D["val"], _D["test"]
    w_sequences = _D.get("sequences_meta")
    print(f"Onbellek: train {len(w_train_pairs)} | val {len(w_val_pairs)} | "
          f"test {len(w_test_pairs)}")
else:
    print("=== TRAIN ===")
    _t0 = time.time()
    _m = motor_kur(train_ids, rng_seed=1234)
    _seq = sekans_uret(_m, train_ids, NUM_SEQUENCES, LOOPS_PER_SEQ, MIN_CHAIN_LEN,
                       "train sekanslari")
    _t_gez = time.time() - _t0
    _t1 = time.time()
    _rec = ciftleri_cikar(_seq, SIM_THRESHOLD)
    w_train_pairs = gommeleri_ekle(_rec, butun_index, rng_seed=7)
    _t_cift = time.time() - _t1
    print(f"  gezinme {_t_gez/60:.1f} dk | cift cikarma {_t_cift/60:.1f} dk")

    print("\n=== VAL ===")
    _t2 = time.time()
    w_val_pairs = degerlendirme_uret(motor_kur(val_ids, 1234), butun_index, val_ids,
                                      VAL_NUM_SEQ, LOOPS_PER_SEQ, MIN_CHAIN_LEN,
                                      SIM_THRESHOLD, VAL_TARGET, "val sekanslari", 43)
    _t_val = time.time() - _t2
    print("\n=== TEST ===")
    _t3 = time.time()
    w_test_pairs = degerlendirme_uret(motor_kur(test_ids, 1234), butun_index, test_ids,
                                       TEST_NUM_SEQ, LOOPS_PER_SEQ, MIN_CHAIN_LEN,
                                       SIM_THRESHOLD, TEST_TARGET, "test sekanslari", 44)
    _t_test = time.time() - _t3

    w_sequences = {"n_train_seq": len(_seq),
                   "dugum": sum(len(c) for c in _seq),
                   "kapsam": len({s.image_id for c in _seq for s in c
                                  if s.image_id} & train_ids),
                   "aday_havuz": ADAY_HAVUZ,
                   # --- sureler (saniye) ---
                   "sure_gezinme": _t_gez, "sure_cift": _t_cift,
                   "sure_val": _t_val, "sure_test": _t_test,
                   "sure_toplam": _t_gez + _t_cift + _t_val + _t_test}
    with open(W_PAIRS_PATH, "wb") as f:
        pickle.dump({"train": w_train_pairs, "val": w_val_pairs,
                     "test": w_test_pairs, "sequences_meta": w_sequences}, f)

# ---------------------------------------------------------------------------
from collections import Counter
_hn = sum(1 for p in w_train_pairs if p["hard_negative_embedding"] is not None)
_md = Counter(p["hard_negative_mode"] for p in w_train_pairs
              if p["hard_negative_mode"] is not None)
print(f"\ntrain {len(w_train_pairs)} cift | hard negative'li {_hn} "
      f"(%{_hn/max(len(w_train_pairs),1)*100:.1f})")
print("hard negative'lerin mod dagilimi: " +
      ", ".join(f"{m} {_md.get(m,0)} (%{_md.get(m,0)/max(_hn,1)*100:.1f})"
                for m in ("color", "shape", "texture")))
if w_sequences:
    print(f"sekans: {w_sequences['n_train_seq']} | dugum {w_sequences['dugum']} | "
          f"train kapsami {w_sequences['kapsam']}/{len(train_ids)} "
          f"(%{w_sequences['kapsam']/len(train_ids)*100:.1f}) | "
          f"aday_havuz {w_sequences.get('aday_havuz')}")
    if "sure_toplam" in w_sequences:
        print(f"\nVERI URETIM SURELERI (bir kez, tum kollar icin ortak):")
        print(f"  train gezinme   {w_sequences['sure_gezinme']/60:>6.1f} dk")
        print(f"  cift cikarma    {w_sequences['sure_cift']/60:>6.1f} dk")
        print(f"  val uretimi     {w_sequences['sure_val']/60:>6.1f} dk")
        print(f"  test uretimi    {w_sequences['sure_test']/60:>6.1f} dk")
        print(f"  {'TOPLAM':<15} {w_sequences['sure_toplam']/60:>6.1f} dk")
print(f"val {len(w_val_pairs)} | test {len(w_test_pairs)}")
_dis = sum(1 for p in w_train_pairs if p["image_id"] not in train_ids)
print(f"train disi kaynak gorsel: {_dis} (0 olmali)")

# yama duzeyi referansi: %17.4 hard negative orani, 4 sinifli mod dagilimi
print("\nyama duzeyi referansi: hard negative orani %17.4")

In [ ]:
# HÜCRE W6: birebir butun duzeyi -- alti kol x bes tohum
# ---------------------------------------------------------------------------
# Mimari, kayiplar ve egitim protokolu HUCRE 5 ile ayni; tek fark dort sinifli
# MODE_MAP (yama duzeyiyle ayni) ve W5'te uretilen ciftler.
# Iki degerlendirme kumesi birlikte raporlanir:
#   yeni : W5'te motorla uretilen test ciftleri (yama duzeyiyle ayni yontem)
#   eski : HUCRE 4'teki test_pairs (her test gorselinden 1 ornek)
import os, time, copy, gc, pickle, random
import numpy as np, torch, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

MODE_MAP4 = {"semantic": 0, "color": 1, "shape": 2, "texture": 3}

class PairDataset4(Dataset):
    def __init__(self, pairs): self.p = pairs
    def __len__(self): return len(self.p)
    def __getitem__(self, i):
        p = self.p[i]
        img = torch.tensor(np.asarray(p["img_embedding"], dtype=np.float32))
        txt = torch.tensor(np.asarray(p["txt_embedding"], dtype=np.float32))
        hn = p.get("hard_negative_embedding"); has = hn is not None
        hn_t = torch.tensor(np.asarray(hn, dtype=np.float32)) if has else torch.zeros_like(img)
        return {"img": img, "txt": txt, "hn": hn_t,
                "has_hn": torch.tensor(1.0 if has else 0.0),
                "mode_idx": torch.tensor(
                    MODE_MAP4.get(p.get("hard_negative_mode") or "semantic", 0),
                    dtype=torch.long)}

# HUCRE 3/4 calistirilmadiysa gereken iki seyi burada kur -- boylece W serisi
# kendi kendine yeterli olur.
if "_cap_emb" not in globals():
    def _cap_emb(iid, rng):
        cid = IID2COCO.get(iid)
        if cid is None or cid not in CAP_EMB: return None
        return np.asarray(rng.choice(CAP_EMB[cid])[1], dtype=np.float32)
    print("_cap_emb burada tanimlandi (HUCRE 4 calistirilmamis)")

if "test_pairs" not in globals():
    # mevcut butun duzeyi kollariyla karsilastirilabilir basit kume:
    # her test gorselinden en fazla 1 ornek (HUCRE 4'teki degerlendirme_ciftleri)
    _r44 = random.Random(44); _tid = sorted(test_ids); _r44.shuffle(_tid)
    test_pairs = []
    for _i in _tid:
        _ce = _cap_emb(_i, _r44)
        if _ce is None: continue
        test_pairs.append({"img_embedding": WHOLE[_i], "txt_embedding": _ce})
        if len(test_pairs) >= 400: break
    print(f"test_pairs burada kuruldu: {len(test_pairs)} cift")

# rastgele kontrol: zincir hic kullanilmadan (butun gorsel, caption)
_rr = random.Random(99)
_ids = sorted(train_ids)
w_random_pairs = []
while len(w_random_pairs) < len(w_train_pairs):
    iid = _rr.choice(_ids); ce = _cap_emb(iid, _rr)
    if ce is None: continue
    w_random_pairs.append({"img_embedding": WHOLE[iid], "txt_embedding": ce,
                           "hard_negative_embedding": None,
                           "hard_negative_mode": None, "image_id": iid})
print(f"rastgele kontrol: {len(w_random_pairs)} cift")

WCKPT = f"{OUT}/butun_birebir_results.pkl"
WSONUC = pickle.load(open(WCKPT, "rb")) if os.path.exists(WCKPT) else {}

def egit_w(varyant, pairs, seed):
    """'vanilla' -> ProjectionHead + plain_info_nce
       'hn_only' -> ProjectionHead + hn_info_nce
       'mode_only' -> SelfInferringHead + plain_info_nce + mod yardimci kaybi
       'full' -> SelfInferringHead + hn_info_nce + mod yardimci kaybi"""
    t0 = time.time(); torch.manual_seed(seed)
    tip = "plain" if varyant in ("vanilla", "hn_only") else "self"
    model = (ProjectionHead() if tip == "plain" else SelfInferringHead(num_modes=4)).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    dl = DataLoader(PairDataset4(pairs), batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    best = copy.deepcopy(model.state_dict()); best_r1 = -1; best_ep = 0; kotu = 0; ep_run = 0
    for ep in range(EPOCHS):
        ep_run = ep + 1; model.train()
        for b in dl:
            img = b["img"].to(device); txt = b["txt"].to(device)
            hn = b["hn"].to(device); has = b["has_hn"].to(device); mi = b["mode_idx"].to(device)
            opt.zero_grad()
            if tip == "plain":
                zi, zt = model(img, txt)
                if varyant == "vanilla":
                    loss = plain_info_nce(zi, zt)
                else:
                    zhn = model.img_proj(hn) if has.sum() > 0 else None
                    loss = hn_info_nce(zi, zt, z_hn=zhn, has_hn=has)
            else:
                mvi, ml = model.infer_mode(img, img); mvt, _ = model.infer_mode(txt, txt)
                zi = model.img_proj(img, mvi); zt = model.txt_proj(txt, mvt)
                if varyant == "mode_only":
                    loss = plain_info_nce(zi, zt)
                else:
                    zhn = model.project_image_with_vec(hn, mvi) if has.sum() > 0 else None
                    loss = hn_info_nce(zi, zt, z_hn=zhn, has_hn=has)
                m = has.bool()
                if m.sum() > 0:
                    loss = loss + MODE_AUX_WEIGHT * F.cross_entropy(ml[m], mi[m])
            loss.backward(); opt.step()
        sch.step()
        if (ep + 1) % EVAL_EVERY == 0 or ep + 1 == EPOCHS:
            r = degerlendir(model, w_val_pairs, tip)["R@1"]
            if r > best_r1: best_r1, best_ep, kotu = r, ep + 1, 0; best = copy.deepcopy(model.state_dict())
            else:
                kotu += 1
                if kotu >= PATIENCE: break
    model.load_state_dict(best)
    return model, tip, {"epochs_run": ep_run, "best_epoch": best_ep,
                        "best_val_r1": best_r1, "elapsed_sec": time.time() - t0}

W_KOLLAR = [
    ("W1 vanilla, rastgele ciftler",        "vanilla",   "random"),
    ("W2 vanilla, ZoomInOut ciftleri",      "vanilla",   "chain"),
    ("W3 yalniz hard negative itme",        "hn_only",   "chain"),
    ("W4 yalniz mod kosullama",             "mode_only", "chain"),
    ("W5 COSMOS ZoomInOut",                 "full",      "chain"),
]

for ad, varyant, kaynak in W_KOLLAR:
    WSONUC.setdefault(ad, {})
    pairs = w_random_pairs if kaynak == "random" else w_train_pairs
    for sd in SEEDS:
        if sd in WSONUC[ad]: continue
        print(f"\n=== {ad} | seed={sd} ===")
        m, tip, meta = egit_w(varyant, pairs, sd)
        meta["yeni"] = degerlendir(m, w_test_pairs, tip)
        meta["eski"] = degerlendir(m, test_pairs, tip)
        WSONUC[ad][sd] = meta
        print(f"  epoch={meta['epochs_run']} sure={meta['elapsed_sec']:.0f}sn  "
              f"R@1 yeni={meta['yeni']['R@1']:.1f}%  eski={meta['eski']['R@1']:.1f}%")
        pickle.dump(WSONUC, open(WCKPT, "wb")); del m; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()

# ---------- dondurulmus temel model ----------
class _Kimlik(torch.nn.Module):
    def forward(self, img, txt): return F.normalize(img, dim=-1), F.normalize(txt, dim=-1)
_f = _Kimlik().to(device)
WSONUC["W0 dondurulmus model"] = {"yeni": degerlendir(_f, w_test_pairs, "plain"),
                                   "eski": degerlendir(_f, test_pairs, "plain")}
pickle.dump(WSONUC, open(WCKPT, "wb"))
print(f"\nTamam -> {WCKPT}")

# ---------- ozet ----------
_d0 = WSONUC["W0 dondurulmus model"]
print(f"\n{'kol':<32}{'R@1 yeni':>16}{'R@1 eski':>16}{'MRR yeni':>16}"
      f"{'dk':>7}{'sn':>7}{'epoch':>7}")
print("-" * 101)
print(f"{'W0 dondurulmus model':<32}{_d0['yeni']['R@1']:>16.1f}"
      f"{_d0['eski']['R@1']:>16.1f}{_d0['yeni']['MRR']:>16.3f}"
      f"{'---':>7}{'---':>7}{'---':>7}")
for ad, _, _ in W_KOLLAR:
    d = WSONUC.get(ad, {})
    d = {k: v for k, v in d.items() if k in SEEDS}
    if len(d) < len(SEEDS):
        print(f"{ad:<32}{f'({len(d)}/{len(SEEDS)})':>16}"); continue
    y = np.array([d[s]["yeni"]["R@1"] for s in SEEDS])
    e = np.array([d[s]["eski"]["R@1"] for s in SEEDS])
    mr = np.array([d[s]["yeni"]["MRR"] for s in SEEDS])
    sn = np.array([d[s]["elapsed_sec"] for s in SEEDS])
    ep = np.array([d[s]["epochs_run"] for s in SEEDS])
    print(f"{ad:<32}{f'{y.mean():.2f} ± {y.std(ddof=1):.2f}':>16}"
          f"{f'{e.mean():.2f} ± {e.std(ddof=1):.2f}':>16}"
          f"{f'{mr.mean():.3f} ± {mr.std(ddof=1):.3f}':>16}"
          f"{sn.mean()/60:>7.1f}{sn.mean():>7.0f}{ep.mean():>7.1f}")
print("-" * 101)
print(f"seed: {SEEDS} | std: ddof=1 | dk/sn/epoch: tohum ortalamasi")
_ws = globals().get("w_sequences")
if _ws and "sure_toplam" in _ws:
    print(f"veri uretimi (bir kez, tum kollar icin ortak): "
          f"{_ws['sure_toplam']/60:.1f} dk")
_tum = 0.0
for ad, _, _ in W_KOLLAR:
    _d = {k: v for k, v in WSONUC.get(ad, {}).items() if k in SEEDS}
    _tum += sum(v["elapsed_sec"] for v in _d.values())
print(f"toplam egitim suresi ({len(W_KOLLAR)} kol x {len(SEEDS)} tohum): "
      f"{_tum/60:.1f} dk")
print("yeni = motorla uretilen test ciftleri (yama duzeyiyle ayni yontem)")
print("eski = her test gorselinden 1 ornek (mevcut butun duzeyi kollariyla karsilastirilabilir)")

# ---------- 2x2 + t-testler ----------
from scipy import stats
def _al(ad, kume="yeni"):
    d = {k: v for k, v in WSONUC.get(ad, {}).items() if k in SEEDS}
    return np.array([d[s][kume]["R@1"] for s in SEEDS]) if len(d) == len(SEEDS) else None
for kume in ("yeni", "eski"):
    A, B, C, D = (_al("W2 vanilla, ZoomInOut ciftleri", kume), _al("W3 yalniz hard negative itme", kume),
                  _al("W4 yalniz mod kosullama", kume), _al("W5 COSMOS ZoomInOut", kume))
    R = _al("W1 vanilla, rastgele ciftler", kume)
    if any(x is None for x in (A, B, C, D, R)): continue
    print(f"\n--- {kume} degerlendirme kumesi ---")
    print(f"{'':<24}{'HN itme YOK':>16}{'HN itme VAR':>16}")
    print(f"{'mod kosullama YOK':<24}{A.mean():>16.2f}{B.mean():>16.2f}")
    print(f"{'mod kosullama VAR':<24}{C.mean():>16.2f}{D.mean():>16.2f}")
    for x, y, et in [(B, A, "HN itme, mod YOKken"), (D, C, "HN itme, mod VARken"),
                     (C, A, "mod kosullama, HN YOKken"), (D, B, "mod kosullama, HN VARken"),
                     (A, R, "ZoomInOut ciftleri vs rastgele")]:
        t, p = stats.ttest_rel(x, y)
        print(f"  {et:<32}{x.mean()-y.mean():+7.2f} puan   t={t:+7.3f}  p={p:.4f}")
    don = WSONUC["W0 dondurulmus model"][kume]["R@1"]
    t, p = stats.ttest_1samp(D, don)
    print(f"  {'COSMOS ZoomInOut vs dondurulmus':<32}{D.mean()-don:+7.2f} puan   "
          f"t={t:+7.3f}  p={p:.4f}")